In [1]:
import os
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.notebook import tqdm
from datetime import time, timedelta

import warnings


In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
warnings.filterwarnings("ignore")
pd.set_option('display.float_format', lambda x: '%.2f' % x)

# functions defintion 

In [3]:
def read_gem_file(gems_path:str, icd9_list_path:str, icd10_list_path: str)-> pd.DataFrame: 
    colspecs = [(0, 5), (6, 13), (14, 19)]
    colnames = ['icd9', 'icd10', 'flags']
    
    gems = pd.read_fwf(gems_path, colspecs=colspecs, names=colnames, dtype=str)
    icd9_list = pd.read_csv(icd9_list_path,dtype=str)
    icd10_list = pd.read_csv(icd10_list_path,dtype=str)
    
    gems['icd9'] = gems['icd9'].str.strip()
    gems['icd10'] = gems['icd10'].str.strip()
    gems[['approximate', 'no_map', 'combination', 'scenario', 'choice_list']] = \
        (gems["flags"].apply(lambda x: pd.Series(list(x))).astype(int))
#     gems = gems[gems.no_map == 0]

    gems.drop(columns=['flags'], inplace=True)
    
    
    icd9_list = dict(zip(icd9_list.icd9, icd9_list.description))
    icd10_list = dict(zip(icd10_list.icd10, icd10_list.description))

    gems['icd9_description'] = gems.icd9.map(icd9_list)
    gems['icd10_description'] = gems.icd10.map(icd10_list)
    
    gems = gems[['icd9','icd9_description',
                 'icd10','icd10_description',
                 'approximate', 'combination', 
                 'scenario','choice_list','no_map']]
    
    return gems

In [4]:
def clean_race(race):
    splits = race.split('//')
    
    if splits[1].startswith('WHITE'):
        splits[1] =  'WHITE'
        
    elif splits[1].startswith('UNABLE'):
        splits[1] =  'UNKNOWN'

    elif splits[1].startswith('BLACK'):
        splits[1] =  'BLACK'

    elif splits[1].startswith('PATIENT'):
        splits[1] =  'UNKNOWN'
        
    elif splits[1].startswith('ASIAN'):
        splits[1] =  'ASIAN'

    elif splits[1].startswith('HISPANIC'):
        splits[1] =  'HISPANIC'

    elif splits[1].startswith('NATIVE'):
        splits[1] =  'NATIVE HAWAIIAN'

    elif splits[1].startswith('AMERICAN INDIAN'):
        splits[1] =  'AMERICAN INDIAN'
        
    return '//'.join(splits)

In [5]:
def clean_outpatient_measurements(
    df: pd.DataFrame,
    pre_post_buffer: pd.Timedelta = timedelta(days=1),
    max_gap_days: int = 30,
    lab_like=("LAB", "MICROBIOLOGY", 'Blood Pressure Standing (1 min)', 'Blood Pressure Lying', 
              'Blood Pressure Sitting', 'BMI (kg/m2)', 'Weight', 'Height', 'Height (Inches)', 'eGFR', 
              'Blood Pressure Standing', 'Weight (Lbs)', 'Blood Pressure Standing (3 mins)', 'BMI', 'Blood Pressure', 
    ),
    admission_code="HOSPITAL_ADMISSION",
    discharge_code="HOSPITAL_DISCHARGE",
) -> pd.DataFrame:
    out = df.copy()
    out["time"] = pd.to_datetime(out["time"], errors="coerce")
    out["code_type"] = out["code"].str.split("//").str[0]
    for c in ("out_id", "er_id"):
        out[c] = np.nan
    out["_order"] = np.arange(len(out))

    # Build visit windows
    admissions = (
        out[out["code"].str.startswith(admission_code, na=False)]
        [["subject_id", "hadm_id", "time"]]
        .rename(columns={"time": "admit_time"})
    )
    discharges = (
        out[out["code"].str.startswith(discharge_code, na=False)]
        [["subject_id", "hadm_id", "time"]]
        .rename(columns={"time": "disch_time"})
    )
    visits = pd.merge(admissions, discharges, on=["subject_id", "hadm_id"], how="inner")

    # 1) Original orphan hadm_id / out_id logic
    orphan = out[out["hadm_id"].isna() & out["code_type"].isin(lab_like)]
    drops = []
    for idx, row in orphan.iterrows():
        sid, t = row["subject_id"], row["time"]
        if pd.isna(t):
            drops.append(idx); continue
        pv = visits[visits["subject_id"] == sid].copy()
        if pv.empty:
            drops.append(idx); continue
        pv["delta"] = (pv["admit_time"] - t).abs()
        nearest = pv.loc[pv["delta"].idxmin()]
        hid, delta = nearest["hadm_id"], nearest["delta"]
        if delta <= pre_post_buffer:
            out.at[idx, "hadm_id"] = hid
        elif delta <= timedelta(days=max_gap_days):
            out.at[idx, "out_id"] = hid
        else:
            drops.append(idx)
    out = out.drop(index=drops)

    # 2) Tag ER and Discharge windows for *all* lab/micro events
    lab_events = out[out["code_type"].isin(lab_like)]
    for idx, row in lab_events.iterrows():
        sid, t = row["subject_id"], row["time"]
        if pd.isna(t):
            continue
        pv = visits[visits["subject_id"] == sid]
        for _, v in pv.iterrows():
            hid, a, d = v["hadm_id"], v["admit_time"], v["disch_time"]
            # ER: before admission
            if a - pre_post_buffer <= t < a:
                out.at[idx, "er_id"] = hid
#                 out.at[idx, "hadm_id"] = hid
            # Discharge: after discharge
            if d < t <= d + pre_post_buffer:
                out.at[idx, "out_id"] = hid
#                 out.at[idx, "hadm_id"] = hid

    # 3) Restore order and return
    out = out.sort_values("_order").drop(columns="_order").reset_index(drop=True)
    return out


In [6]:
def push_procedure_and_sort(df: pd.DataFrame) -> pd.DataFrame:
    """
    Adjust PROCEDURE timestamps and properly order events per patient, visit-aware.
    
    Steps:
    - Push PROCEDURE timestamps at midnight to 23:59:59
    - Ensure RACE/GENDER always at the top of each patient's sequence
    - Sort events by:
        1. subject_id
        2. static (True first)
        3. earliest time per hadm_id (to respect admission order)
        4. event time within visits
    """
    df = df.copy()
    
    # Fix PROCEDURE timestamps at 00:00:00 → end of day
    mask_procedure = df["code"].str.startswith("PROCEDURE")
    mask_midnight = df["time"].dt.time == time(0, 0)
    df.loc[mask_procedure & mask_midnight, "time"] = (
        df.loc[mask_procedure & mask_midnight, "time"].dt.normalize() +
        pd.Timedelta(hours=23, minutes=59, seconds=59)
    )
    
    # Mark static tokens like RACE and GENDER
    df["is_static"] = df["code"].str.startswith(("RACE", "GENDER"))
    
    # Compute earliest time per hadm_id per patient to get admission order
    admission_order = (
        df[~df["hadm_id"].isna()]
        .groupby(["subject_id", "hadm_id"])["time"]
        .min()
        .reset_index()
        .rename(columns={"time": "hadm_order_time"})
    )
    
    df = df.merge(admission_order, on=["subject_id", "hadm_id"], how="left")

    # NaN hadm_id (outpatient) → set hadm_order_time = time (so it's positioned by actual time)
    df["hadm_order_time"] = df["hadm_order_time"].fillna(df["time"])

    # Final sort: static first, then visit order, then time
    df = df.sort_values(
        by=["subject_id", "is_static", "hadm_order_time", "time"],
        ascending=[True, False, True, True]
    ).drop(columns=["is_static", "hadm_order_time"]).reset_index(drop=True)

    return df

In [7]:
def make_mapping_decesion(decision1,decision2):
    if decision1 == decision2:
        return decision1
    elif decision1 != decision2:
        if type(decision2) == float:
            return decision1[:3]
        elif decision1[:3] == decision2[:3]:
            return decision1[:3]
        else:
            return decision2[:3]

In [8]:
def build_best_icd10_mapper(gems_df: pd.DataFrame) -> dict:
    """
    Build best ICD-9 → ICD-10 mapping from GEMs file, prioritizing:
    - no_map == 0 (valid map)
    - combination == 0 (single-code map)
    - approximate == 0 (exact match preferred)
    - lowest choice_list (preferred target)

    Assumes ICD-9 codes are already in the same format as the target dataset.
    """
    df = gems_df.copy()

    # Filter for valid, standalone, mappable entries
    df = df[(df["no_map"] == 0) & (df["combination"] == 0)]

    # Ranking mechanism: prioritize exact match, low choice_list, shorter code
    df["rank"] = (
        df["approximate"] * 100 +
        df["choice_list"] * 10 +
        df["icd10"].str.len()
    )

    # Choose best ICD-10 per ICD-9
    best = (
        df.sort_values(["icd9", "rank"])
        .drop_duplicates(subset="icd9", keep="first")
    )

    # No padding applied to ICD-9 codes
    return dict(zip(best["icd9"].astype(str), best["icd10"].astype(str)))

In [9]:
mimic_data_path = os.path.join('..','data','MEDS_output','data','train')
mimic_metadata_path = os.path.join('..','data','raw','MEDS_output','metadata',)
labs_metadata_path = os.path.join('..','resources','mimic-mapping')
labs_dimension_path = os.path.join('..','resources','mimic-mapping')

icd_mapping_files_path = os.path.join('..','resources','icd-code-conversion')
medications_files_path = os.path.join('..','resources','medications')
labs_files_path = os.path.join('..','resources','labs')

In [10]:
# shard = pd.read_parquet(os.path.join(mimic_data_path,'0.parquet'))
# 323

medgemma_rankings_d = pd.read_csv(os.path.join(icd_mapping_files_path,'1-2-many_gems_d_ranked1.csv'))
gpt_rankings_d = pd.read_csv(os.path.join(icd_mapping_files_path,'1-2-many_gems_d_ranked_gpt1.csv'))

medgemma_rankings_p = pd.read_csv(os.path.join(icd_mapping_files_path,'1-2-many_gems_p_ranked1.csv'),dtype={"icd9_code": str, "icd10_code": str})
gpt_rankings_p = pd.read_csv(os.path.join(icd_mapping_files_path,'1-2-many_gems_p_ranked_gpt1.csv'),dtype={"icd9_code": str, "icd10_code": str})

gems_cm_labeled = pd.read_csv(os.path.join(icd_mapping_files_path,'gems_cm_labeled.csv'))
high_level_d = pd.read_csv(os.path.join(icd_mapping_files_path,'high_level_d.csv'))

gems_pcs_labeled = pd.read_csv(os.path.join(icd_mapping_files_path,'gems_pcs_labeled.csv'),dtype={"icd9": str, "icd10": str})
high_level_p = pd.read_csv(os.path.join(icd_mapping_files_path,'high_level_p.csv'),dtype={"icd9": str, "icd10": str})

cleaned_medications = pd.read_csv(os.path.join(medications_files_path,'cleaned_medications.csv'))

labs_metadata = pd.read_csv(os.path.join(labs_metadata_path,'d_labitems_to_loinc.csv'))
labs_dimension = pd.read_csv(os.path.join(labs_dimension_path,'d_labitems.csv')) 
cleaned_lab_values = pd.read_csv(os.path.join(labs_files_path,'lab_textual_mapping.csv'))

icu_items_dimensions = pd.read_csv(os.path.join(labs_metadata_path,'d_items.csv'))


In [11]:
# Patient without hospital admission
# patients_without_hadm = []
# for pid in tqdm(shard.subject_id.unique()):
#     patient = shard[shard.subject_id == pid]
#     patient_hadm_id = patient.hadm_id.unique()
#     if patient_hadm_id.shape[0] == 1:
#         patients_without_hadm.append(int(pid))

# # filter
# shard = shard[shard.subject_id.isin(patients_without_hadm) == False].reset_index(drop=True)    

In [12]:
# remove table name from code
# shard['table'] = shard.code.apply(lambda x: x.split('//')[-1])
# shard.code = shard.code.apply(lambda x: '//'.join(x.split('//')[:-1]))

In [13]:
# # handle patient race
# # unify races
# shard['race'] = shard.code.apply(lambda x: x.split('//')[1] if x.startswith('RACE') else np.nan)
# shard['code'] = shard.code.apply(lambda x: clean_race(x) if x.startswith('RACE') else x)

# # Step 1: Create base sequential 'filter' column
# shard["filter"] = range(len(shard))

# # Step 2: Identify RACE rows
# race_mask = shard["code"].str.contains("RACE", na=False)

# # Step 3: Assign the same filter value for consecutive RACE rows
# filter_values = []
# group_id = -1
# for i, is_race in tqdm(enumerate(race_mask)):
#     if i == 0 or not is_race or not race_mask.iloc[i - 1]:
#         group_id += 1
#     filter_values.append(group_id)
# shard["filter"] = filter_values

# # Step 4: Collapse duplicates by keeping first row per filter group
# shard = shard.groupby("filter", as_index=False).first()
# shard.drop(columns=['filter'],inplace=True)
# patients_with_multiple_races = []
# for pid in tqdm(shard.subject_id.unique()):
#     patient = shard[shard.subject_id == pid]
#     patient_races = patient[patient.code.str.startswith('RACE')]
#     if patient_races.shape[0] > 2:
#         patients_with_multiple_races.append(int(pid))
        
# print(len(patients_with_multiple_races))

In [14]:
# # clean outpatient measuerments
# cleaned_timelines = []
# for pid in tqdm(shard.subject_id.unique()):
#     patient = shard[shard.subject_id == pid]
#     cleaned_patient = clean_outpatient_measurements(patient)
#     cleaned_timelines.append(cleaned_patient)

# shard = pd.concat(cleaned_timelines).reset_index(drop=True)
# del(cleaned_timelines)

In [15]:
# # clean empty admissions
# empty_hadms = []
# patients_with_empty_hadms = []

# for pid in tqdm(shard.subject_id.unique()):
#     patient = shard[shard.subject_id == pid]
#     admissions = patient.hadm_id.dropna().unique()

#     for hid in admissions:
#         admission = patient[patient.hadm_id == hid].reset_index(drop=True)
#         age_rows = admission[admission.code.str.startswith('AGE_AT_ADMISSION')]

#         if not age_rows.empty:
#             idx = age_rows.index[0]
#             if idx + 1 < len(admission):
#                 next_code = admission.iloc[idx + 1].code
#                 if next_code.startswith('DISCHARGE-FROM-HOSPITAL'):
#                     empty_hadms.append(int(hid))
#                     patients_with_empty_hadms.append(int(pid))

# patients_with_single_empty = []
# for pid in patients_with_empty_hadms:
#     patient = shard[shard.subject_id == pid]
#     hids = patient.hadm_id.unique()
# #     print(hids.shape[0],pid)
#     if hids.shape[0] == 2:
#         patients_with_single_empty.append(pid)
        
# patients_with_empty_hadms = [pid for pid in patients_with_empty_hadms if \
#                              pid not in patients_with_single_empty]

# shard = shard[shard.subject_id.isin(patients_with_single_empty) == False].reset_index(drop=True)

# shard = shard[(shard.hadm_id.isin(empty_hadms) == False) & 
#               (shard.out_id.isin(empty_hadms) == False)].reset_index(drop=True)

In [16]:
# handle procedure codes
# shard = push_procedure_and_sort(shard)
# shard.subject_id.unique()

In [17]:
# # get first rankings only
# medgemma_rankings_d = medgemma_rankings_d.groupby('icd9_code').first().reset_index()
# gpt_rankings_d = gpt_rankings_d.groupby('icd9_code').first().reset_index()

# # drop rank and reason column
# medgemma_rankings_d = medgemma_rankings_d.iloc[:,:-2]
# gpt_rankings_d = gpt_rankings_d.iloc[:,:-2]

# #merge both tables
# merged_ranking_d = pd.merge(gpt_rankings_d,medgemma_rankings_d,how='outer',on='icd9_code',suffixes=('_gpt','_medgemma'))

# # apply final mapping 
# merged_ranking_d['final'] = merged_ranking_d.apply(lambda x: make_mapping_decesion(x.icd10_code_gpt,x.icd10_code_medgemma),axis=1)

In [18]:
# # exact mapping
# exact_d = gems_cm_labeled[gems_cm_labeled.label == 'exact']
# exact_mappings_d = dict(zip(exact_d['icd9'],exact_d['icd10']))

# # one to one mapping
# one_to_one_d = gems_cm_labeled[gems_cm_labeled.label == 'one_to_one']
# one_to_one_mappings_d = dict(zip(one_to_one_d['icd9'],one_to_one_d['icd10']))

# # one to many mapping
# one_to_many_mappings_d = dict(zip(merged_ranking_d['icd9_code'],merged_ranking_d['final']))

# # high level mapping
# high_level_mapping_d = dict(zip(high_level_d.icd9,high_level_d.icd10))

# all_mappings = exact_mappings_d | one_to_one_mappings_d | one_to_many_mappings_d | high_level_mapping_d

In [19]:
# # all possible mapping
# shard['icd9_to_icd10_d'] = shard.diag_icd_code.map(all_mappings)

# # combination mapping
# combinations = gems_cm_labeled[gems_cm_labeled.label == 'combination']
# combinations = combinations[combinations.scenario == 1]
# combinations.groupby(['icd9','choice_list']).first().reset_index()
# combinations = combinations[['icd9','icd10']]
# shard = shard.merge(combinations,left_on='diag_icd_code',right_on='icd9',how='left')
# shard.icd9_to_icd10_d = shard.apply(lambda x: x['icd10'] if pd.notna(x['icd10']) else x['icd9_to_icd10_d'], axis=1)

# # no map elimination
# no_map_code = gems_cm_labeled[gems_cm_labeled.label == 'no_map'].icd9.unique()
# shard = shard[shard.diag_icd_code.isin(no_map_code) == False].reset_index(drop=True)

# shard = shard[shard.diag_icd_code.isin(['V451','V854','V138','V51','V127','V109','V581','V608','V152','V251','V122',
#                                                 'V610','V155','V403','V135']) == False].reset_index(drop=True)

# # unify names
# shard.code = shard.apply(lambda x:'//'.join([x.code_type,x.icd9_to_icd10_d]) if x.code.startswith('DIAGNOSIS-ICD//9') else x.code,axis=1)
# shard.code = shard.apply(lambda x:'//'.join([x.code_type,x.diag_icd_code]) if x.code.startswith('DIAGNOSIS-ICD//10') else x.code,axis=1)

# shard.drop(columns=['icd9','icd10'],inplace=True)

In [20]:
# a = shard[shard.code.str.startswith('DIAGNOSIS-ICD//9')]
# a[a.icd9_to_icd10_d.isna()]

In [21]:
# # procedure codes mapping
# # get first rankings only
# medgemma_rankings_p = medgemma_rankings_p.groupby('icd9_code').first().reset_index()
# gpt_rankings_p = gpt_rankings_p.groupby('icd9_code').first().reset_index()

# # drop rank and reason column
# medgemma_rankings_p = medgemma_rankings_p.iloc[:,:-2]
# gpt_rankings_p = gpt_rankings_p.iloc[:,:-2]

# #merge both tables
# merged_ranking_p = pd.merge(gpt_rankings_p,medgemma_rankings_p,how='outer',on='icd9_code',suffixes=('_gpt','_medgemma'))

# # apply final mapping 
# merged_ranking_p['final'] = merged_ranking_p.apply(lambda x: make_mapping_decesion(x.icd10_code_gpt,x.icd10_code_medgemma),axis=1)


In [22]:
# # exact mapping
# exact_p = gems_pcs_labeled[gems_pcs_labeled.label == 'exact']
# exact_mappings_p = dict(zip(exact_p['icd9'],exact_p['icd10']))

# # one to one mapping
# one_to_one_p = gems_pcs_labeled[gems_pcs_labeled.label == 'one_to_one']
# one_to_one_mappings_p = dict(zip(one_to_one_p['icd9'],one_to_one_p['icd10']))

# # one to many mapping
# one_to_many_mappings_p = dict(zip(merged_ranking_p['icd9_code'],merged_ranking_p['final']))

# # high level mapping
# high_level_mapping_p = dict(zip(high_level_p.icd9,high_level_p.icd10))

# all_mappings = exact_mappings_p | one_to_one_mappings_p | one_to_many_mappings_p | high_level_mapping_p

In [23]:
# # all possible mapping
# shard['icd9_to_icd10_p'] = shard.proc_icd_code.map(all_mappings)

# # combination mapping
# combinations = gems_pcs_labeled[gems_pcs_labeled.label == 'combination']
# combinations = combinations[combinations.scenario == 1]
# combinations.groupby(['icd9','choice_list']).first().reset_index()
# combinations = combinations[['icd9','icd10']]
# shard = shard.merge(combinations,left_on='proc_icd_code',right_on='icd9',how='left')
# shard.icd9_to_icd10_p = shard.apply(lambda x: x['icd10'] if pd.notna(x['icd10']) else x['icd9_to_icd10_p'], axis=1)

# # no map elimination
# no_map_code = gems_pcs_labeled[gems_pcs_labeled.label == 'no_map'].icd9.unique()
# shard = shard[shard.proc_icd_code.isin(no_map_code) == False].reset_index(drop=True)

# shard = shard[shard.proc_icd_code.isin(['857']) == False].reset_index(drop=True)

# # unify names
# shard.code = shard.apply(lambda x:'//'.join([x.code_type,x.icd9_to_icd10_p]) if x.code.startswith('PROCEDURE-ICD//9') else x.code,axis=1)
# shard.code = shard.apply(lambda x:'//'.join([x.code_type,x.proc_icd_code]) if x.code.startswith('PROCEDURE-ICD//10') else x.code,axis=1)

# shard.drop(columns=['icd9','icd10'],inplace=True)

In [24]:
# b = shard[shard.code.str.startswith('PROCEDURE-ICD//9')]
# b[b.icd9_to_icd10_p.isna()]

In [25]:
# # Medications cleaning
# cleaned_medications = cleaned_medications.replace(np.nan,None)
# medication_mapping = dict(zip(cleaned_medications.original_name,cleaned_medications.clean))
# shard['clean_medication'] = shard.medication.map(medication_mapping)
# shard = shard[shard.clean_medication.isna() == False]
# shard.code = shard.apply(lambda x:'//'.join([x.code_type,x.clean_medication]) if x.code.startswith('MEDICATION') else x.code,axis=1)
# shard = shard[shard.code.str.startswith('MEDICATION//UNK') == False].reset_index(drop=True)

In [26]:
# b = shard[shard.code.str.startswith('MEDICATION')]
# shard[shard.clean_medication.isna() == False]

In [27]:
# shard.shape[0] -1500362

In [28]:
# # Microbiology cleaning
# shard.code = shard.apply(lambda x:'//'.join([x.code_type, str(int(x.micro_test_itemid)),x.micro_test_name]) if x.code.startswith('MICROBIOLOGY') else x.code,axis=1)
# shard.micro_org_name = shard.micro_org_name.apply(lambda x: None if x == 'NEGATIVE' else x)
# shard.micro_org_name = shard.micro_org_name.apply(lambda x: None if x == 'NO GROWTH' else x)
# shard.micro_spec_type_desc = shard.micro_spec_type_desc.apply(lambda x: 'TISSUE' if x == 'XXX' else x)
# shard.micro_spec_type_desc = shard.micro_spec_type_desc.apply(lambda x: 'BLOOD CULTURE' if x == '' else x)
# shard = shard[shard.micro_org_name != 'CANCELLED'].reset_index(drop=True)
# shard.text_value = shard.apply(lambda x: 'NEGATIVE' if (x.micro_org_name == None) & (x.code_type == 'MICROBIOLOGY') else x.text_value, axis=1)

In [29]:
# not_labs = [50807, 50812, 50829, 50845, 50886, 50887, 50888, 50897, 50919,
#             50923, 50932, 50933, 50934, 50947, 50955, 50979, 50984, 50985,
#             51038, 51056, 51103, 51107, 51129, 51571, 51591, 51599, 51600,
#             51601, 51602, 51603, 51604, 51608, 51612, 51671, 51678, 51698,
#             51699, 51700, 51702, 51703, 51706, 51712, 51717, 51718, 51719,
#             51720, 51727, 51752, 51757, 51759, 51760, 51771, 51796, 51806,
#             51827, 51828, 51830, 51831, 51839, 51901, 51905, 51906, 51907,
#             51924, 51953, 51955, 51978, 51993, 51995, 51997, 51998, 52014,
#             52016, 52023, 52025, 52033, 52036, 52043, 52066, 52067, 52068,
#             52118, 52161, 52186, 52195, 52229, 52230, 52231, 52232, 52233,
#             52234, 52235, 52236, 52237, 52238, 52239, 52240, 52241, 52242,
#             52243, 52244, 52245, 52246, 52247, 52248, 52249, 52250, 52251,
#             52252, 52253, 52254, 52287, 52288, 52289, 52290, 52313, 52314,
#             52315, 52334, 52370, 52371, 52372, 52374, 52392, 52393, 52405,
#             52406, 52412, 52415, 52418, 52419, 52420, 52421, 52422, 52423,
#             53127, 51564, 51597, 51605, 51657, 51658, 51659, 51660, 51661, 
#             51663, 51664, 51665, 51686, 51732, 51733, 51734, 51735, 51736,
#             51737, 51762, 51763, 51764, 51765, 51766, 51767, 51768, 51772,
#             51789, 51817, 51849, 51850, 51851, 51852, 51856, 51857, 51902,
#             51903, 51904, 51908, 51909, 51916, 51939, 51954, 51956, 51970,
#             51971, 51973, 52004, 52005, 52006, 52007, 52008, 52009, 52010,
#             52011, 52012, 52018, 52019, 52020, 52021, 52080, 52081, 52083,
#             52084, 52110, 52136, 52137, 52147, 52148, 52153, 52169, 52191,
#             52194, 52215, 52217, 52317, 52318, 52333, 52394, 52395, 52396,
#             52397, 52398, 52399, 52400, 52401, 52402, 52424, 52425, 52426,
#             52427, 53122, 51662, 50827, 50828, 51509,51513]

In [30]:
# # Handle labs
# labs_metadata = labs_metadata.rename(columns={'itemid (omop_source_code)':'itemid'})




# lab_label = dict(zip(labs_dimension['itemid'], labs_dimension['label']))
# lab_fluid = dict(zip(labs_dimension['itemid'], labs_dimension['fluid']))
# lab_category = dict(zip(labs_dimension['itemid'], labs_dimension['category']))
# lab_description = dict(zip(labs_metadata['itemid'], labs_metadata['omop_concept_name']))
# lab_frequency = dict(zip(labs_metadata['itemid'], labs_metadata['labevents_row_count']))
# lab_valueuom = dict(zip(labs_metadata['itemid'], labs_metadata['valueuom']))


# shard['lab_label'] =  shard.lab_itemid.map(lab_label)
# shard['lab_fluid'] =  shard.lab_itemid.map(lab_fluid)
# shard['lab_category'] =  shard.lab_itemid.map(lab_category)
# shard['lab_description'] =  shard.lab_itemid.map(lab_description)
# shard['lab_frequency'] =  shard.lab_itemid.map(lab_frequency)
# shard['lab_valueuom'] =  shard.lab_itemid.map(lab_valueuom)

# shard.lab_label = shard.lab_label.apply(lambda x: x.split(', ')[0] if type(x) == str 
#                                         and ((x.split(', ')[-1] in list(labs_metadata.fluid.unique())) 
#                                         or (x.split(', ')[-1] in ['Body Fluid', 'Other Fluid'])) 
#                                         else x)


# shard = shard[shard.lab_itemid.isin(not_labs) == False].reset_index(drop=True)
# all_values = shard[shard.code_type == 'LAB'].text_value.unique()
# target_values = cleaned_lab_values.original_values.unique()
# values = []
# for value in all_values:
#     if value not in target_values:
#         values.append(value)
# values = [item for item in values if item is not None]
# shard = shard[shard.text_value.isin(values) == False].reset_index(drop=True)

# cleaned = dict(zip(cleaned_lab_values.original_values, cleaned_lab_values.clean))
# shard.text_value = shard.text_value.apply(lambda x: cleaned[x] if x in cleaned.keys() else x)

# numeric = cleaned_lab_values[cleaned_lab_values.numric.notna()]
# cleaned_numeric = dict(zip(numeric.clean,numeric.numric))
# shard.numeric_value = shard.apply(lambda x: cleaned_numeric[x.text_value] if (x.text_value in cleaned_numeric.keys()) & (x.code_type == 'LAB') else x.numeric_value,axis=1)
# shard.text_value = shard.apply(lambda x: None if (x.text_value in cleaned_numeric.keys()) & (x.code_type == 'LAB') else x.text_value, axis=1)

# shard.text_value = shard.apply(lambda x: 'UNKNOWN' if ((pd.isna(x.numeric_value) and pd.isna(x.text_value))
#                                                    or
#                                                       (pd.isna(x.numeric_value) and x.text_value == '___')) 
#                                                    and
#                                                       (x.code_type == 'LAB')
#                                                    else 
#                                                        x.text_value, axis=1)

# shard.code = shard.apply(lambda x: '//'.join([x.code,x.lab_fluid,x.lab_label]) if x.code.startswith('LAB//') else x.code,axis= 1)



In [31]:
# # # handle ICU procedures
# shard = shard[shard.category != '7-Communication'].reset_index(drop=True)
# shard.code = shard.apply(lambda x:'//'.join([x.code_type,str(int(x.itemid)),x.abbreviation]) if x.code.startswith('ICU-PROCEDURE') else x.code,axis=1)

In [32]:
# # ICU fluids_output processing
# shard.numeric_value = shard.apply(lambda x: abs(x.numeric_value) if x.code_type == 'ICU-FLUID-OUTPUT' else x.numeric_value, axis=1)
# shard.code = shard.apply(lambda x:'//'.join([x.code_type,str(int(x.itemid)),x.abbreviation]) if x.code_type == 'ICU-FLUID-OUTPUT' else x.code,axis=1)

In [33]:
# # handle infusions
# shard.code = shard.apply(lambda x:'//'.join([x.code_type,str(int(x.itemid)),x.abbreviation]) if x.code_type == 'ICU-INFUSION' else x.code,axis=1)
# shard.numeric_value = shard.apply(lambda x: x.amount if x.code_type == 'ICU-INFUSION' else x.numeric_value,axis=1)

In [34]:
# handle ICU chart 
# shard.code = shard.apply(lambda x:'//'.join([x.code_type,str(int(x.itemid)),x.abbreviation]) if x.code_type == 'ICU-CHART' else x.code,axis=1)

In [35]:
# shard['seq_id'] = shard.apply(lambda x: next((x[col] for col in ['out_id', 'er_id', 'hadm_id'] if pd.notna(x[col])), np.nan),axis=1)

In [36]:
# shard['value'] = shard.apply(lambda x: x.numeric_value if pd.notna(x.numeric_value) else x.text_value,axis=1)

In [37]:
# columns = ['subject_id', 'seq_id', 'out_id', 'er_id', 'hadm_id', 'icustay_id', 'disch_id', 'time', 'code', 
#            'numeric_value', 'text_value', 'itemid', 'died_in_hosp', 'icu_los', 'admission_type',
#            'admission_location', 'discharge_location', 'diag_version', 'diag_icd_code', 'diag_seq_num', 
#            'drg_severity', 'drg_mortality',  'drg_type', 'drg_code', 'priority', 'specimen_id', 
#            'lab_lower_limit', 'lab_upper_limit', 'lab_flag', 'lab_unit', 'lab_itemid', 'gender', 
#            'route', 'frequency', 'doses_per_24_hrs', 'medication', 'proc_seq_num', 'proc_version',
#            'proc_icd_code', 'micro_specimen_id', 'micro_org_name', 'micro_test_name', 'micro_spec_type_desc', 
#            'micro_test_itemid', 'icu_care_unit',  'category', 'label', 'abbreviation', 'rate', 'unit', 
#            'amount', 'amountuom', 'ordercategorydescription', 'ordercategoryname','secondaryordercategoryname', 
#            'ordercomponenttypedescription', 'table', 'race', 'code_type', 'icd9_to_icd10_d', 'icd9_to_icd10_p',
#            'clean_medication', 'lab_label', 'lab_fluid', 'lab_category', 'lab_description', 'lab_frequency']

# shard = shard[columns]

In [38]:
# shard.hadm_id = shard.apply(lambda x: np.nan if (x.hadm_id == x.er_id) or (x.hadm_id == x.disch_id) else x.hadm_id, axis= 1)

In [39]:
# shard.out_id = shard.apply(lambda x: np.nan if x.out_id == x.disch_id else x.out_id, axis= 1)

In [40]:
# shard.disch_id =  shard.apply(lambda x: np.nan if x.er_id != x.disch_id else x.disch_id,axis = 1)

In [41]:
# def create_pairs_from_list(input_list):
#     if not input_list:
#         return []

#     # Check if the list has an odd number of elements
#     if len(input_list) % 2 != 0:
#         # If odd, the last element is handled separately
#         pairs = list(zip(input_list[:-1:2], input_list[1::2]))
#         pairs.append(input_list[-1])  # Add the last element as a single item
#     else:
#         # If even, create pairs from the entire list
#         pairs = list(zip(input_list[::2], input_list[1::2]))
#     return pairs

# names = list(range(365))
# names = [(str(name) + '.parquet') for name in names]
# pairs = create_pairs_from_list(names)
# len(pairs[-1])

In [42]:
# interm_path = os.path.join('..','data','intermediate')

In [43]:
# num = 0
# for pair in tqdm(pairs):
#     if len(pair) == 2:
        
#         shard_1 = pd.read_parquet(os.path.join(interm_path,pair[0]))
#         shard_2 = pd.read_parquet(os.path.join(interm_path,pair[1]))
        
#         shard_1.out_id = shard_1.apply(lambda x: np.nan if x.out_id == x.disch_id else x.out_id, axis= 1)
#         shard_2.out_id = shard_2.apply(lambda x: np.nan if x.out_id == x.disch_id else x.out_id, axis= 1)
        
#         shard_1.text_value = shard_1.apply(lambda x: 'UNKNOWN' if x.numeric_value == 999999.00 else x.text_value, axis = 1)
#         shard_2.text_value = shard_1.apply(lambda x: 'UNKNOWN' if x.numeric_value == 999999.00 else x.text_value, axis = 1)
        
#         shard_1.numeric_value = shard_1.numeric_value.apply(lambda x: np.nan if x == 999999.00 else x)
#         shard_2.numeric_value = shard_2.numeric_value.apply(lambda x: np.nan if x == 999999.00 else x)
        
#         shard = pd.concat([shard_1,shard_2],ignore_index=True)
        
#         shard.to_parquet(os.path.join(processed_path,f'{num}.parquet')
    
#     elif len(pair) > 2:
        
#         shard_1 = pd.read_parquet(os.path.join(interm_path,pair))
#         shard_1.out_id = shard_1.apply(lambda x: np.nan if x.out_id == x.disch_id else x.out_id, axis= 1)
#         shard_1.text_value = shard_1.apply(lambda x: 'UNKNOWN' if x.numeric_value == 999999.00 else x.text_value, axis = 1)
#         shard_1.numeric_value = shard_1.numeric_value.apply(lambda x: np.nan if x == 999999.00 else x)
#         shard_1.to_parquet(os.path.join(processed_path,f'{num}.parquet')
                         
#     num+=1
        

In [44]:
# shard[shard.code.str.startswith('ED')]

In [45]:
# ICU
# shard[shard.icustay_id == 36957970].head(50)

In [46]:
#HADM
# shard[shard.hadm_id == 10017886]

In [47]:
# def clean_outpatient_measurements(
#     df: pd.DataFrame,
#     pre_post_buffer: pd.Timedelta = timedelta(days=1),
#     max_gap_days: int = 30,
#     lab_like=("LAB", "MICROBIOLOGY", 'Blood Pressure Standing (1 min)', 'Blood Pressure Lying', 
#               'Blood Pressure Sitting', 'BMI (kg/m2)', 'Weight', 'Height', 'Height (Inches)', 'eGFR', 
#               'Blood Pressure Standing', 'Weight (Lbs)', 'Blood Pressure Standing (3 mins)', 'BMI', 'Blood Pressure', 
#               'ED_REGISTRATION','ED_OUT'),
    
#     admission_code="HOSPITAL_ADMISSION",
#     discharge_code="HOSPITAL_DISCHARGE",) -> pd.DataFrame:
    
#     out = df.copy()
#     out["time"] = pd.to_datetime(out["time"], errors="coerce")
#     out["code_type"] = out["code"].str.split("//").str[0]
#     for c in ("out_id", "er_id"):
#         out[c] = np.nan
#     out["_order"] = np.arange(len(out))

#     # Build visit windows
#     admissions = (
#         out[out["code"].str.startswith(admission_code, na=False)]
#         [["subject_id", "hadm_id", "time"]]
#         .rename(columns={"time": "admit_time"})
#     )
#     discharges = (
#         out[out["code"].str.startswith(discharge_code, na=False)]
#         [["subject_id", "hadm_id", "time"]]
#         .rename(columns={"time": "disch_time"})
#     )
#     visits = pd.merge(admissions, discharges, on=["subject_id", "hadm_id"], how="inner")

#     # 1) Original orphan hadm_id / out_id logic
#     orphan = out[out["hadm_id"].isna() & out["code_type"].isin(lab_like)]
#     drops = []
#     for idx, row in orphan.iterrows():
#         sid, t = row["subject_id"], row["time"]
#         if pd.isna(t):
#             drops.append(idx); continue
#         pv = visits[visits["subject_id"] == sid].copy()
#         if pv.empty:
#             drops.append(idx); continue
#         pv["delta"] = (pv["admit_time"] - t).abs()
#         nearest = pv.loc[pv["delta"].idxmin()]
#         hid, delta = nearest["hadm_id"], nearest["delta"]
#         if delta <= pre_post_buffer:
#             out.at[idx, "hadm_id"] = hid
#         elif delta <= timedelta(days=max_gap_days):
#             out.at[idx, "out_id"] = hid
#         else:
#             drops.append(idx)
#     out = out.drop(index=drops)

#     # 2) Tag ER and Discharge windows for *all* lab/micro events
#     lab_events = out[out["code_type"].isin(lab_like)]
#     for idx, row in lab_events.iterrows():
#         sid, t = row["subject_id"], row["time"]
#         if pd.isna(t):
#             continue
#         pv = visits[visits["subject_id"] == sid]
#         for _, v in pv.iterrows():
#             hid, a, d = v["hadm_id"], v["admit_time"], v["disch_time"]
#             # ER: before admission
#             if a - pre_post_buffer <= t < a:
#                 out.at[idx, "er_id"] = hid
# #                 out.at[idx, "hadm_id"] = hid
#             # Discharge: after discharge
#             if d < t <= d + pre_post_buffer:
#                 out.at[idx, "out_id"] = hid
# #                 out.at[idx, "hadm_id"] = hid

#     # 3) Restore order and return
#     out = out.sort_values("_order").drop(columns="_order").reset_index(drop=True)
#     return out


In [190]:
import os
import polars as pl
pl.Config.set_tbl_rows(1000)

polars.config.Config

In [326]:
import polars as pl


OMR_EVENTS = {
    "Blood Pressure Standing (1 min)",
    "Blood Pressure Lying",
    "Blood Pressure Sitting",
    "BMI (kg/m2)",
    "Weight",
    "Height",
    "Height (Inches)",
    "eGFR",
    "Blood Pressure Standing",
    "Weight (Lbs)",
    "Blood Pressure Standing (3 mins)",
    "BMI",
    "Blood Pressure",
}


def segment_care_stage(df: pl.DataFrame) -> pl.DataFrame:

    # chronological order
    df = (
        df
        .sort("time")
        .with_row_index("event_idx")
    )

    care_stages = []

    current_stage = "OUTPATIENT"

    discharge_date = None


    for row in df.iter_rows(named=True):

        code = row["code"]
        hadm_id = row["hadm_id"]
        event_time = row["time"]


        # -------------------------
        # Static events
        # -------------------------
        if row["event_idx"] < 2:

            care_stages.append(None)
            continue


        # -------------------------
        # Close inpatient after
        # discharge day ends
        # -------------------------
        if discharge_date is not None and event_time is not None:

            if event_time.date() > discharge_date:

                if current_stage in ["ED", "INPATIENT", "ICU"]:
                    current_stage = "OUTPATIENT"

                discharge_date = None


        # -------------------------
        # ED start
        # -------------------------
        if code.startswith("ED_REGISTRATION"):

            current_stage = "ED"
            discharge_date = None


        # -------------------------
        # Hospital admission
        # -------------------------
        elif code.startswith("HOSPITAL_ADMISSION"):

            current_stage = "INPATIENT"
            discharge_date = None


        # -------------------------
        # ICU admission
        # -------------------------
        elif code.startswith("ICU_ADMISSION"):

            current_stage = "ICU"


        # -------------------------
        # ICU discharge
        # -------------------------
        elif code.startswith("ICU_DISCHARGE"):

            current_stage = "INPATIENT"


        # -------------------------
        # Hospital discharge
        # Keep inpatient for
        # discharge-day events
        # -------------------------
        elif code.startswith("HOSPITAL_DISCHARGE"):

            current_stage = "INPATIENT"

            if event_time is not None:
                discharge_date = event_time.date()


        # -------------------------
        # OMR events
        # Only switch to outpatient
        # if not inside active encounter
        # -------------------------
        elif code in OMR_EVENTS:

            if current_stage not in ["ED", "INPATIENT", "ICU"]:

                current_stage = "OUTPATIENT"


        # -------------------------
        # Other events
        # Do not modify stage
        # -------------------------
        else:

            pass


        care_stages.append(current_stage)


    return (
        df
        .with_columns(
            pl.Series("care_stage", care_stages)
        )
    )

In [341]:
import polars as pl


def assign_visit_id(df: pl.DataFrame) -> pl.DataFrame:

    df = df.sort("time")

    visit_ids = []

    current_visit = 0

    in_acute = False
    last_outpatient_date = None


    for row in df.iter_rows(named=True):

        code = row["code"]
        stage = row["care_stage"]
        event_time = row["time"]


        # -------------------------
        # Static
        # -------------------------
        if stage == None:
            visit_ids.append(None)
            continue


        # -------------------------
        # New ED visit
        # -------------------------
        if code.startswith("ED_REGISTRATION"):

            current_visit += 1
            in_acute = True

            visit_ids.append(current_visit)
            continue


        # -------------------------
        # New inpatient visit
        # only if not already in ED
        # -------------------------
        if code.startswith("HOSPITAL_ADMISSION"):

            if not in_acute:
                current_visit += 1

            in_acute = True

            visit_ids.append(current_visit)
            continue


        # -------------------------
        # ICU stays belong to same visit
        # -------------------------
        if stage == "ICU":

            visit_ids.append(current_visit)
            continue


        # -------------------------
        # Remaining inpatient/ED
        # -------------------------
        if stage in ["ED", "INPATIENT"]:

            visit_ids.append(current_visit)
            continue


        # -------------------------
        # Outpatient visits
        # grouped by calendar date
        # -------------------------
        if stage == "OUTPATIENT":

            event_date = (
                event_time.date()
                if event_time is not None
                else None
            )

            if in_acute:
                current_visit += 1
                in_acute = False
                last_outpatient_date = event_date

            elif event_date != last_outpatient_date:

                current_visit += 1
                last_outpatient_date = event_date


            visit_ids.append(current_visit)
            continue


        visit_ids.append(current_visit)


    return (
        df
        .with_columns(
            pl.Series(
                "visit_id",
                visit_ids,
                dtype=pl.Int32
            )
        )
    )

In [342]:
def within(duration: float):

    if duration <= 0:
        return '0-D'

    # exact days
    elif duration > 0 and duration <= 1:
        return '1-D'
    elif duration > 1 and duration <= 2:
        return '2-D'
    elif duration > 2 and duration <= 3:
        return '3-D'
    elif duration > 3 and duration <= 4:
        return '4-D'
    elif duration > 4 and duration <= 5:
        return '5-D'
    elif duration > 5 and duration <= 6:
        return '6-D'
    elif duration > 6 and duration <= 7:
        return '7-D'

    # weeks
    elif duration > 7 and duration <= 14:
        return '2-W'
    elif duration > 14 and duration <= 21:
        return '3-W'
    elif duration > 21 and duration <= 28:
        return '4-W'

    # months
    elif duration > 28 and duration <= 60:
        return '2-M'
    elif duration > 60 and duration <= 90:
        return '3-M'
    elif duration > 90 and duration <= 120:
        return '4-M'
    elif duration > 120 and duration <= 150:
        return '5-M'
    elif duration > 150 and duration <= 180:
        return '6-M'
    elif duration > 180 and duration <= 210:
        return '7-M'
    elif duration > 210 and duration <= 240:
        return '8-M'
    elif duration > 240 and duration <= 270:
        return '9-M'
    elif duration > 270 and duration <= 300:
        return '10-M'
    elif duration > 300 and duration <= 330:
        return '11-M'
    elif duration > 330 and duration <= 360:
        return '12-M'

    else:
        return '1-Y+'

In [347]:
def add_time_tokens(df: pl.DataFrame) -> pl.DataFrame:

    df = (
        df
        .sort("time")
        .with_columns(
            (
                pl.col("time")
                .diff()
                .dt.total_seconds()
                / (24 * 3600)
            )
            .alias("time_diff")
        )
    )

    rows = []

    previous_visit = None
    previous_visit_end = None
    first_clinical_event = True

    for row in df.iter_rows(named=True):

        visit_id = row["visit_id"]
        care_stage = row["care_stage"]
        current_time = row["time"]

        # keep static tokens
        if row["code"].startswith(("GENDER", "MEDS_BIRTH")):

            rows.append(row)

            # use MEDS_BIRTH as temporal anchor
            if row["time"] is not None:
                previous_visit_end = row["time"]

            continue


        # first clinical event after static section
        if first_clinical_event:

            if (
                current_time is not None 
                and previous_visit_end is not None
            ):

                duration = (
                    current_time - previous_visit_end
                ).total_seconds() / (24 * 3600)

                bucket = within(duration)

                token = row.copy()

                token["code"] = f"TIME-GAP//{bucket}"

                token["care_stage"] = None
                token["visit_id"] = None
                token["hadm_id"] = None

                if "code_type" in token:
                    token["code_type"] = "TIME-GAP"

                if "text_value" in token:
                    token["text_value"] = bucket

                if "numeric_value" in token:
                    token["numeric_value"] = duration

                if "time_diff" in token:
                    token["time_diff"] = None

                rows.append(token)

            first_clinical_event = False


        # normal visit transition
        if previous_visit is not None and visit_id != previous_visit:

            if (
                current_time is not None 
                and previous_visit_end is not None
            ):

                duration = (
                    current_time - previous_visit_end
                ).total_seconds() / (24 * 3600)

                bucket = within(duration)

                token = row.copy()

                token["code"] = f"TIME-GAP//{bucket}"
                token["care_stage"] = None
                token["visit_id"] = None
                token["hadm_id"] = None

                if "code_type" in token:
                    token["code_type"] = "TIME-GAP"

                if "text_value" in token:
                    token["text_value"] = bucket

                if "numeric_value" in token:
                    token["numeric_value"] = duration

                if "time_diff" in token:
                    token["time_diff"] = None

                rows.append(token)


        rows.append(row)

        previous_visit = visit_id
        previous_visit_end = current_time


    return pl.DataFrame(rows, schema=df.schema, strict=False)

In [351]:
import polars as pl


def add_token_type(df: pl.DataFrame) -> pl.DataFrame:

    def get_token_type(code):

        if code is None:
            return None

        parts = code.split("//")

        # LAB
        if parts[0] == "LAB":
            if len(parts) > 1:
                if parts[1] == "SPECIMEN_COLLECTED":
                    return "LAB_SPECIMEN"

                elif parts[1] == "RESULT":
                    return "LAB_RESULT"

                else:
                    return "ICU_CHART_EVENT"


        # PROCEDURE
        if parts[0] == "PROCEDURE":

            if len(parts) > 1:
                if parts[1] == "ICD":
                    return "PROCEDURE_ICD"

                elif parts[1] == "START":
                    return "ICU_PROCEDURE_START"

                elif parts[1] == "END":
                    return "ICU_PROCEDURE_END"


        # DIAGNOSIS
        if parts[0] == "DIAGNOSIS":

            if len(parts) > 1 and parts[1] == "ICD":
                return "DIAGNOSIS_ICD"


        # INFUSION
        if parts[0] == "INFUSION_START":
            return "ICU_INFUSION_START"

        if parts[0] == "INFUSION_END":
            return "ICU_INFUSION_END"


        # FLUID OUTPUT
        if parts[0] == "SUBJECT_FLUID_OUTPUT":
            return "ICU_SUBJECT_FLUID_OUTPUT"


        # MEDICATION
        if parts[0] == "MEDICATION":

            if len(parts) > 1:

                if parts[1] == "START":
                    return "MEDICATION_START"

                elif parts[1] == "STOP":
                    return "MEDICATION_STOP"


        return parts[0]

    
    df = df.with_columns(
        pl.col("code")
        .map_elements(
            get_token_type,
            return_dtype=pl.String
        )
        .alias("code_type")
    )
    
    df = (
    df
    .drop("event_idx", strict=False)
    .with_row_index("event_idx")
    )
    return df

In [386]:
def process_omr_numeric(df: pl.DataFrame) -> pl.DataFrame:

    bp_parts = (
        pl.col("text_value")
        .str.split("/")
    )

    bp_sys = (
        bp_parts
        .list.get(0, null_on_oob=True)
        .cast(pl.Float64, strict=False)
    )

    bp_dia = (
        bp_parts
        .list.get(1, null_on_oob=True)
        .cast(pl.Float64, strict=False)
    )

    return df.with_columns(
        pl.when(
            pl.col("code").is_in(OMR_EVENTS)
            & pl.col("code").str.contains("Blood Pressure")
        )
        .then(
            (bp_sys + 2 * bp_dia) / 3
        )

        .when(
            pl.col("code").is_in(OMR_EVENTS)
        )
        .then(
            pl.col("text_value")
            .cast(pl.Float64, strict=False)
        )

        .otherwise(
            pl.col("numeric_value")
        )
        .alias("numeric_value")
    )

In [408]:
shard = pl.read_parquet(os.path.join(mimic_data_path,'1.parquet')) 
shard = shard.filter(pl.col('code').str.starts_with('HCPCS') == False)
shard = process_omr_numeric(shard)


#['event_idx','time','code','numeric_value','text_value','hadm_id','care_stage','visit_id','time_diff','code_type',]

In [409]:
processed = []
for subject_id in tqdm(shard['subject_id'].unique()):
    
    timeline = shard.filter(pl.col('subject_id') == subject_id)
    timeline = segment_care_stage(timeline)
    timeline = assign_visit_id(timeline)
    timeline = add_time_tokens(timeline)
    timeline = add_token_type(timeline)
    processed.append(timeline)
    
a = pl.concat(processed)    

  0%|          | 0/999 [00:00<?, ?it/s]

In [412]:
a

event_idx,subject_id,time,code,numeric_value,text_value,insurance,language,marital_status,race,hadm_id,drg_severity,drg_mortality,emar_id,emar_seq,priority,route,frequency,doses_per_24_hrs,poe_id,icustay_id,order_id,link_order_id,unit,ordercategorydescription,statusdescription,care_stage,visit_id,time_diff,code_type
u32,i64,datetime[μs],str,f64,str,str,str,str,str,i64,i64,i64,str,i64,str,str,str,i64,str,i64,i64,i64,str,str,str,str,i32,f64,str
0,10001667,null,"""GENDER//F""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""GENDER"""
1,10001667,2087-01-01 00:00:00,"""MEDS_BIRTH""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""MEDS_BIRTH"""
2,10001667,2173-08-22 01:40:00,"""TIME-GAP//1-Y+""",31644.069444,"""1-Y+""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""TIME-GAP"""
3,10001667,2173-08-22 01:40:00,"""ED_REGISTRATION""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""ED""",1,31644.069444,"""ED_REGISTRATION"""
4,10001667,2173-08-22 01:40:00,"""TRANSFER_TO//ED//Emergency Dep…",null,null,null,null,null,null,22672901,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""ED""",1,0.0,"""TRANSFER_TO"""
5,10001667,2173-08-22 01:50:00,"""LAB//SPECIMEN_COLLECTED//50868""",null,null,null,null,null,null,null,null,null,null,null,"""STAT""",null,null,null,null,null,null,null,null,null,null,"""ED""",1,0.006944,"""LAB_SPECIMEN"""
6,10001667,2173-08-22 01:50:00,"""LAB//SPECIMEN_COLLECTED//50882""",null,null,null,null,null,null,null,null,null,null,null,"""STAT""",null,null,null,null,null,null,null,null,null,null,"""ED""",1,0.0,"""LAB_SPECIMEN"""
7,10001667,2173-08-22 01:50:00,"""LAB//SPECIMEN_COLLECTED//50902""",null,null,null,null,null,null,null,null,null,null,null,"""STAT""",null,null,null,null,null,null,null,null,null,null,"""ED""",1,0.0,"""LAB_SPECIMEN"""
8,10001667,2173-08-22 01:50:00,"""LAB//SPECIMEN_COLLECTED//50912""",null,null,null,null,null,null,null,null,null,null,null,"""STAT""",null,null,null,null,null,null,null,null,null,null,"""ED""",1,0.0,"""LAB_SPECIMEN"""


In [ ]:
# 
# nums = []
# for x in range(365):
#     p_num = pd.read_parquet(f'../data/intermediate/{x}.parquet').subject_id.unique().shape[0]
#     nums.append(p_num)

In [ ]:
# out = a[a.out_id==x]
# out.shape

# emer = a[a.er_id==x]
# emer.shape

# hadm = a[a.hadm_id==x]
# hadm.shape

# disch = a[a.disch_id==x]
# disch.shape

# # outpatient before admission
# out[out.time< hadm.time.iloc[0]].shape

# # outpatient after admission
# out[out.time> hadm.time.iloc[-1]].shape

In [89]:
gems_cm[gems_cm.icd9.str.startswith('V127')]

,icd9,icd9_description,icd10,icd10_description,approximate,combination,scenario,choice_list,no_map
21993,V1270,Personal history of unspecified digestive disease,Z8719,Personal history of other diseases of the digestive system,1,0,0,0,0
21994,V1271,Personal history of peptic ulcer disease,Z8711,Personal history of peptic ulcer disease,0,0,0,0,0
21995,V1272,Personal history of colonic polyps,Z86010,Personal history of colonic polyps,0,0,0,0,0
21996,V1279,Personal history of other diseases of digestive system,Z8719,Personal history of other diseases of the digestive system,1,0,0,0,0


In [83]:
icd_cm_mapping_path = os.path.join('..','resources','icd-code-conversion','diagnosis','diagnosis_gems_2018','2018_I9gem.txt')
icd_pcs_mapping_path = os.path.join('..','resources','icd-code-conversion','procedure','procedure_gems_2018','gem_i9pcs.txt')

icd9_d_lists_path = os.path.join('..','resources','code-lists','diagnosis','icd-9-d-code-list.csv')
icd10_d_lists_path = os.path.join('..','resources','code-lists','diagnosis','icd-10-d-code-list.csv')

icd9_p_lists_path = os.path.join('..','resources','code-lists','procedure','icd-9-p-code-list.csv')
icd10_p_lists_path = os.path.join('..','resources','code-lists','procedure','icd-10-p-code-list.csv')


In [84]:
gems_cm = read_gem_file(icd_cm_mapping_path,icd9_d_lists_path,icd10_d_lists_path)
gems_pcs = read_gem_file(icd_pcs_mapping_path,icd9_p_lists_path,icd10_p_lists_path)


In [85]:
# over all dataset information collection
diag_codes = []
proc_codes = []
for file in tqdm(os.listdir(mimic_data_path)):
    part = pd.read_parquet(os.path.join(mimic_data_path,file))
    diag = list(part[part.diag_version == 9].diag_icd_code.unique())
    proc = list(part[part.proc_version == 9].proc_icd_code.unique())
    diag_codes.extend(diag) 
    proc_codes.extend(proc) 
    
diag_codes = list(set(diag_codes))
proc_codes = list(set(proc_codes))

  0%|          | 0/365 [00:00<?, ?it/s]

In [90]:
# Diagnosis codes labeling
exact = []
approximate = []
no_map = []
high_level_only = []
for code in tqdm(diag_codes):
    if code in gems_cm[gems_cm.approximate == 0].icd9.unique():
        exact.append(code)
    elif code in gems_cm[(gems_cm.approximate == 1) & (gems_cm.no_map == 0)].icd9.unique():
        approximate.append(code)
    elif code in gems_cm[(gems_cm.approximate == 1) & (gems_cm.no_map == 1)].icd9.unique():
        no_map.append(code)
    else:
        high_level_only.append(code)


print('exact matching=',len(exact))
print('approximate matching=',len(approximate))
print('no_map=',len(no_map))
print('high-level-only=',len(high_level_only))

print('total_retrived=',  len(exact) + len(approximate) + len(no_map) + len(high_level_only))
print('total_exists=', len(diag_codes))



# gems_cm[gems_cm.icd9.str.startswith(tuple(high_level_only))]
gems_cm_exact = gems_cm[(gems_cm.icd9.isin(exact)) & (gems_cm.approximate == 0)]
gems_cm_no_map = gems_cm[(gems_cm.icd9.isin(no_map))]
gems_cm_approximate = gems_cm[(gems_cm.icd9.isin(approximate))]

comb_codes = gems_cm_approximate[gems_cm_approximate.combination == 1].icd9.unique()
non_comb = gems_cm_approximate[gems_cm_approximate.icd9.isin(comb_codes)==False]
temp = non_comb[non_comb.combination == 0].groupby('icd9').count().reset_index()
one_to_one_codes = temp[temp.icd10 ==1].icd9.unique()
one_to_many_codes = temp[temp.icd10 >1].icd9.unique()

combination = gems_cm_approximate[gems_cm_approximate.icd9.isin(comb_codes)]
one_to_one = gems_cm_approximate[gems_cm_approximate.icd9.isin(one_to_one_codes)]
one_to_many = gems_cm_approximate[gems_cm_approximate.icd9.isin(one_to_many_codes)]


gems_cm_exact[['label']] = 'exact'
gems_cm_no_map[['label']] = 'no_map'
combination[['label']] = 'combination'
one_to_one[['label']] = 'one_to_one'
one_to_many[['label']] = 'one_to_many'

# gems_cm_labeled = pd.concat([gems_cm_exact,gems_cm_no_map,combination,one_to_one,one_to_many],ignore_index=True)

  0%|          | 0/9143 [00:00<?, ?it/s]

exact matching= 2440
approximate matching= 6344
no_map= 284
high-level-only= 75
total_retrived= 9143
total_exists= 9143


In [91]:
gems_cm_labeled

In [26]:
# b = gems_cm[gems_cm.icd9.str.startswith(tuple(high_level_only))]
# b['high_level'] = b.icd9.apply(lambda x: max([r for r in high_level_only if x.startswith(r)], key=len, default=None))
# b = b.groupby('high_level').first().head(60).reset_index()
# b = b.drop(columns='icd9')
# b = b.rename(columns={'high_level':'icd9'})
# b.icd10 = b.icd10.apply(lambda x: x[:3])

In [27]:
# b = b.groupby('high_level').first().head(60).reset_index()
# b = b.drop(columns='icd9')
# b = b.rename(columns={'high_level':'icd9'})
# b.icd10 = b.icd10.apply(lambda x: x[:3])

In [28]:
# b.to_csv('../resources/icd-code-conversion/high_level_d.csv',index=False)

In [29]:
# one_to_many[one_to_many.icd9.isin(a)].to_csv('gpt_d.csv',index= False)

In [30]:
# c = pd.read_csv('./1-2-many_gems_d_ranked.csv')

In [31]:
# gpt_rankings_d.icd9_code.unique().shape

In [32]:
# d = pd.concat((medgemma_rankings_d,c),ignore_index=True)

In [33]:
# z = []
# for code in one_to_many_codes:
#     if code not in c.icd9_code.unique():
#         z.append(code)

In [34]:
# gems_cm[gems_cm.icd9.isin(z)].to_csv('./medgemma_d.csv')

In [35]:
# a = pd.read_csv('./1-2-many_gems_d_ranked.csv')

In [36]:
# z

In [37]:
# b = pd.read_csv('../resources/icd-code-conversion/1-2-many_gems_d_ranked1.csv')

In [38]:
# c = pd.concat((b,a),ignore_index=True)

In [39]:
# c.to_csv('../resources/icd-code-conversion/1-2-many_gems_d_ranked1.csv')

In [40]:
# f = gems_cm[gems_cm.icd9.isin(z)]
# f['final'] = f.icd10.apply(lambda x: x[:3])

In [41]:
# f

In [42]:
# f.to_csv('./medgemma_d.csv',index=False)

In [310]:
# # procerdure codes lableing 
# exact = []
# approximate = []
# no_map = []
# high_level_only = []
# for code in tqdm(proc_codes):
#     if code in gems_pcs[gems_pcs.approximate == 0].icd9.unique():
#         exact.append(code)
#     elif code in gems_pcs[(gems_pcs.approximate == 1) & (gems_pcs.no_map == 0)].icd9.unique():
#         approximate.append(code)
#     elif code in gems_pcs[(gems_pcs.approximate == 1) & (gems_pcs.no_map == 1)].icd9.unique():
#         no_map.append(code)
#     else:
#         high_level_only.append(code)
        
# print('exact matching=',len(exact))
# print('approximate matching=',len(approximate))
# print('no_map=',len(no_map))
# print('high-level-only=',len(high_level_only))

# print('total_retrived=',  len(exact) + len(approximate) + len(no_map) + len(high_level_only))
# print('total_exists=', len(proc_codes))

# gems_pcs_exact = gems_pcs[(gems_pcs.icd9.isin(exact)) & (gems_pcs.approximate == 0)]
# gems_pcs_no_map = gems_pcs[(gems_pcs.icd9.isin(no_map))]
# gems_pcs_approximate = gems_pcs[(gems_pcs.icd9.isin(approximate))]

# comb_codes = gems_pcs_approximate[gems_pcs_approximate.combination == 1].icd9.unique()
# non_comb = gems_pcs_approximate[gems_pcs_approximate.icd9.isin(comb_codes)==False]
# temp = non_comb[non_comb.combination == 0].groupby('icd9').count().reset_index()
# one_to_one_codes = temp[temp.icd10 ==1].icd9.unique()
# one_to_many_codes = temp[temp.icd10 >1].icd9.unique()

# combination = gems_pcs_approximate[gems_pcs_approximate.icd9.isin(comb_codes)]
# one_to_one = gems_pcs_approximate[gems_pcs_approximate.icd9.isin(one_to_one_codes)]
# one_to_many = gems_pcs_approximate[gems_pcs_approximate.icd9.isin(one_to_many_codes)]

# gems_pcs_exact[['label']] = 'exact'
# gems_pcs_no_map[['label']] = 'no_map'
# combination[['label']] = 'combination'
# one_to_one[['label']] = 'one_to_one'
# one_to_many[['label']] = 'one_to_many'

# gems_pcs_labeled = pd.concat([gems_pcs_exact,gems_pcs_no_map,combination,one_to_one,one_to_many],ignore_index=True)

In [71]:
# b = gems_cm[gems_cm.icd9.str.startswith(tuple(high_level_only))]
# b['high_level'] = b.icd9.apply(lambda x: max([r for r in high_level_only if x.startswith(r)], key=len, default=None))
# b = b.groupby('high_level').first().head(60).reset_index()
# b = b.drop(columns='icd9')
# b = b.rename(columns={'high_level':'icd9'})
# b.icd10 = b.icd10.apply(lambda x: x[:3])

In [309]:
# combinations

In [308]:
# b.to_csv('../resources/icd-code-conversion/high_level_p.csv')

In [32]:
# a = []

# for code in one_to_many_codes:
#     if code not in medgemma_rankings_p.icd9_code.unique():
#         a.append(code)
# pd.read_csv('./1-2-many_gems_p_ranked_gpt.csv',dtype={"icd9_code": str, "icd10_code": str}).icd9_code.unique().shape

In [33]:
# pd.read_csv('../resources/icd-code-conversion/1-2-many_gems_p_ranked.csv',dtype={"icd9_code": str, "icd10_code": str})

In [307]:
# a = pd.read_csv('/scratch/sas10092/ehr-foundation/notebooks/1-2-many_gems_p_ranked.csv',dtype={"icd9_code": str, "icd10_code": str})

In [306]:
# a.icd9_code.unique().shape

In [152]:
# a.to_csv('../resources/icd-code-conversion/1-2-many_gems_p_ranked1.csv',index=False)

In [305]:
# b = pd.read_csv('../resources/icd-code-conversion/1-2-many_gems_p_ranked1.csv',dtype={"icd9_code": str, "icd10_code": str})

In [304]:
# b.icd9_code.unique().shape

In [302]:
# c = pd.concat((b,a),ignore_index=True)

In [303]:
# c.icd9_code.unique().shape

In [157]:
# c.to_csv('../resources/icd-code-conversion/1-2-many_gems_p_ranked1.csv',index=False)

In [301]:
# z = []

# for code in one_to_many_codes:
#     if code not in c.icd9_code.unique():
#         z.append(code)

In [159]:
# len(z)

2

In [160]:
# gems_pcs[gems_pcs.icd9.isin(z)].to_csv('medgemma_p.csv',index=False)

In [300]:
# gems_pcs[gems_pcs.icd9.isin(z)]

In [16]:
# gems_cm = gems_cm[gems_cm.no_map == 0]
# exact = gems_cm[gems_cm.approximate == 0].icd9.unique()
# approximate = gems_cm[gems_cm.approximate == 1].icd9.unique()
# data_codes = shard[shard.diag_version == 9].diag_icd_code.unique()

# print(data_codes.shape,'\n')

# x=0 
# for code in exact:
#     if code in data_codes:
#         x+=1
        
# print('exact: ',x,(x/data_codes.shape[0])*100)

# y=0 
# for code in approximate:
#     if code in data_codes:
#         y+=1
        
# print('approximate: ',y,(y/data_codes.shape[0])*100)

# z=0
# for code in data_codes:
#     if code not in np.concat((approximate,exact)):
#         z+=1

# print('non: ',z,(z/data_codes.shape[0])*100)

# print('\n')

# a = gems_cm[(gems_cm.approximate == 1) & (gems_cm.combination == 0)]
# b = a.groupby(['icd9']).count().sort_values('icd10',ascending=False)

# one_2_many = b[b.icd10 >1].reset_index().icd9.unique()

# x = 0
# for code in one_2_many:
#     if code in data_codes:
#         x+=1
        
# print('1-to-many-no-combination: ',x,(x/y)*100)

# # a = gems_cm[(gems_cm.approximate == 1) & (gems_cm.combination == 0)]
# # b = a.groupby(['icd9']).count().sort_values('icd10',ascending=False)

# # one_2_many = b[b.icd10 ==1].reset_index().icd9.unique()

# # x = 0
# # for code in one_2_many:
# #     if code in data_codes:
# #         x+=1
        
# # print('1-to-one-no-combination: ',x,(x/y)*100)


# # a = gems_cm[(gems_cm.approximate == 1) & (gems_cm.combination == 1)]
# # b = a.groupby(['icd9']).count().sort_values('icd10',ascending=False)

# # one_2_many = b[b.icd10 >1].reset_index().icd9.unique()

# # x = 0
# # for code in one_2_many:
# #     if code in data_codes:
# #         x+=1
        
# # print('combination: ',x,(x/y)*100)

In [17]:
# gems_pcs = gems_pcs[gems_pcs.no_map == 0]

# exact = gems_pcs[gems_pcs.approximate == 0].icd9.unique()
# approximate = gems_pcs[gems_pcs.approximate == 1].icd9.unique()
# data_codes = shard[shard.proc_version == 9].proc_icd_code.unique()

# print(data_codes.shape)

# x=0 
# for code in exact:
#     if code in data_codes:
#         x+=1
        
# print('exact: ',x,(x/data_codes.shape[0])*100)

# y=0 
# for code in approximate:
#     if code in data_codes:
#         y+=1
        
# print('approximate: ',y,(y/data_codes.shape[0])*100)

# z=0
# for code in data_codes:
#     if code not in np.concat((approximate,exact)):
#         z+=1

# print('non: ',z,(z/data_codes.shape[0])*100)

# print('\n')

# a = gems_pcs[(gems_pcs.approximate == 1) & (gems_pcs.combination == 0)]
# b = a.groupby(['icd9']).count().sort_values('icd10',ascending=False)

# one_2_many = b[b.icd10 >1].reset_index().icd9.unique()

# x = 0
# for code in one_2_many:
#     if code in data_codes:
#         x+=1
        
# print('1-to-many-no-combination: ',x,(x/y)*100)

# # a = gems_pcs[(gems_pcs.approximate == 1) & (gems_pcs.combination == 0)]
# # b = a.groupby(['icd9']).count().sort_values('icd10',ascending=False)

# # one_2_many = b[b.icd10 ==1].reset_index().icd9.unique()

# # x = 0
# # for code in one_2_many:
# #     if code in data_codes:
# #         x+=1
        
# # print('1-to-one-no-combination: ',x,(x/y)*100)


# # a = gems_pcs[(gems_pcs.approximate == 1) & (gems_pcs.combination == 1)]
# # b = a.groupby(['icd9']).count().sort_values('icd10',ascending=False)

# # one_2_many = b[b.icd10 >1].reset_index().icd9.unique()

# # x = 0
# # for code in one_2_many:
# #     if code in data_codes:
# #         x+=1
        
# # print('combination: ',x,(x/y)*100)

In [218]:
# import pandas as pd
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.metrics.pairwise import cosine_similarity


# df = b
# # Adjust to use 'icd9' and 'icd10' columns
# expected_cols = ["icd9", "icd9_description", "icd10", "icd10_description"]
# missing = [c for c in expected_cols if c not in df.columns]
# if missing:
#     raise ValueError(f"Missing expected column(s): {missing}")

# # Prepare texts
# all_texts = pd.concat([df["icd9_description"].fillna(""), df["icd10_description"].fillna("")], ignore_index=True)
# vectorizer = TfidfVectorizer(stop_words="english")
# tfidf_matrix = vectorizer.fit_transform(all_texts)

# # We'll compute similarity per row with new transform to ensure unique icd9 vector reused
# result_rows = []
# for icd9_code, grp in df.groupby("icd9", sort=False):
#     icd9_desc = grp.iloc[0]["icd9_description"] if pd.notnull(grp.iloc[0]["icd9_description"]) else ""
#     icd9_vec = vectorizer.transform([icd9_desc])
#     icd10_vecs = vectorizer.transform(grp["icd10_description"].fillna(""))
#     sims = cosine_similarity(icd9_vec, icd10_vecs).flatten()
#     grp = grp.copy()
#     grp["similarity"] = sims
#     grp = grp.sort_values("similarity", ascending=False)
#     grp["rank"] = range(1, len(grp)+1)
#     result_rows.append(grp)

# ranked_df = pd.concat(result_rows, ignore_index=True)
# output_path = "/mnt/data/one_to_many_ranked_similarity.csv"
# ranked_df.to_csv(output_path, index=False)

# Save the ranked results
# output_path = "/mnt/data/one_to_many_ranked_similarity.csv"
# ranked_df.to_csv(output_path, index=False)

# Show a preview



In [299]:
# ranked_df.groupby('icd9').first()

In [ ]:
# # handle diagnosis codes mapping
# gems_cm = gems_cm[gems_cm.no_map != 1]
# icd9_to_icd10_cm = build_best_icd10_mapper(gems_cm)

# # map exact approximate
# shard['icd9_to_icd10'] = shard.diag_icd_code.map(icd9_to_icd10_cm)

# # map combinations
# gems_cm_comb = gems_cm[(gems_cm.combination == 1)& (gems_cm.scenario == 1)].iloc[:,:2]
# shard = shard.merge(gems_cm_comb,left_on='diag_icd_code',right_on='icd9',how='left')
# shard.icd9_to_icd10 = shard.apply(lambda x: x['icd10'] if pd.notna(x['icd10']) else x['icd9_to_icd10'], axis=1)

# # remove unmatched
# all_icd9 = shard[shard.diag_version == 9]
# unmapped_icd9 = all_icd9[all_icd9.icd9_to_icd10.isna()].code.unique()
# shard = shard[shard.code.isin(unmapped_icd9) == False].reset_index(drop=True)
# # unify names
# shard.code = shard.apply(lambda x:'//'.join([x.code_type,x.icd9_to_icd10]) if x.code.startswith('DIAGNOSIS-ICD//9') else x.code,axis=1)
# shard.code = shard.apply(lambda x:'//'.join([x.code_type,x.diag_icd_code]) if x.code.startswith('DIAGNOSIS-ICD//10') else x.code,axis=1)

# shard.drop(columns=['icd9','icd10'],inplace=True)

In [18]:
# gems_pcs = gems_pcs[gems_pcs.no_map != 1]
# icd9_to_icd10_pcs = build_best_icd10_mapper(gems_pcs)

# # map exact approximate
# shard['icd9_to_icd10'] = shard.proc_icd_code.map(icd9_to_icd10_pcs)

# # map combinations
# gems_pcs_comb = gems_pcs[(gems_pcs.combination == 1)& (gems_pcs.scenario == 1)].iloc[:,:2]
# shard = shard.merge(gems_pcs_comb,left_on='proc_icd_code',right_on='icd9',how='left')
# shard.icd9_to_icd10 = shard.apply(lambda x: x['icd10'] if pd.notna(x['icd10']) else x['icd9_to_icd10'], axis=1)

# # remove unmatched
# all_icd9 = shard[shard.proc_version == 9]
# unmapped_icd9 = all_icd9[all_icd9.icd9_to_icd10.isna()].code.unique()
# shard = shard[shard.code.isin(unmapped_icd9) == False].reset_index(drop=True)

# # unify names
# shard.code = shard.apply(lambda x:'//'.join([x.code_type,x.icd9_to_icd10]) if x.code.startswith('PROCEDURE-ICD//9') else x.code,axis=1)
# shard.code = shard.apply(lambda x:'//'.join([x.code_type,x.proc_icd_code]) if x.code.startswith('PROCEDURE-ICD//10') else x.code,axis=1)

# shard.drop(columns=['icd9','icd10'],inplace=True)

In [19]:
# stage_tokens = pd.DataFrame(columns=['subject_id','hadm_id','icu_id','outpatient','emergency','discharge','inpatient_icu','icu_stay'])
# stage_durations = pd.DataFrame(columns=['subject_id','hadm_id','icu_id','outpatient','emergency','discharge','inpatient_icu','icu_stay'])

# for pid in tqdm(shard.subject_id.unique()):
#     patient = shard[shard.subject_id == pid]
#     patient['time'] = pd.to_datetime(patient["time"], errors="coerce")
#     hids = patient.hadm_id.unique()
#     hids = hids[np.isnan(hids) == False]
#     for hid in hids:
#         admission = patient[(patient.hadm_id == hid) |  
#                             (patient.out_id == hid) ]

        
#         outpatient_tokens = admission[admission.out_id == hid]
#         try:
#             outpatient_duration = (outpatient_tokens['time'].iloc[-1] - outpatient_tokens['time'].iloc[0]).total_seconds()/(86400)
#         except:
#             outpatient_duration = np.nan

#         emeregency_tokens = admission[admission.er_id == hid]
#         try:
#             emeregency_duration = (emeregency_tokens['time'].iloc[-1] - emeregency_tokens['time'].iloc[0]).total_seconds()/(86400)
#         except:
#             emeregency_duration = np.nan
        
#         discharge_tokens = admission[admission.disch_id == hid]
#         try:
#             discharge_duration = (discharge_tokens['time'].iloc[-1] - discharge_tokens['time'].iloc[0]).total_seconds()/(86400)
#         except:
#             discharge_duration = np.nan
        
#         inpatient_stay = admission[admission.hadm_id == hid]
#         inpatient_tokens = abs(inpatient_stay.shape[0] - emeregency_tokens.shape[0] - discharge_tokens.shape[0])
        
#         inpatient_stay = inpatient_stay.loc[inpatient_stay[inpatient_stay.code.str.startswith('ADMISSION-AT-HOSPITAL')].index[0]:
#                                             inpatient_stay[inpatient_stay.code.str.startswith('DISCHARGE-FROM-HOSPITAL')].index[0]]
#         try:
#             inpatient_duration = (inpatient_stay['time'].iloc[-1] - inpatient_stay['time'].iloc[0]).total_seconds()/(86400)
#         except:
#             inpatient_duration = np.nan
            
#         icuids = admission.icustay_id.unique()
#         icuids = icuids[np.isnan(icuids) == False]

#         if icuids.shape[0] == 0:
#             icuid = np.nan
#             icu_tokens = np.nan
#             icu_duration = np.nan
#             tokens = {'subject_id':pid,
#                        'hadm_id':hid,
#                        'icu_id':icuid,
#                        'outpatient':outpatient_tokens.shape[0],
#                        'emergency':emeregency_tokens.shape[0],
#                        'discharge':discharge_tokens.shape[0],
#                        'inpatient_icu':inpatient_tokens,
#                        'icu_stay':icu_tokens}
#             stage_tokens.loc[len(stage_tokens)] = tokens
#             durations = {'subject_id':pid,
#                        'hadm_id':hid,
#                        'icu_id':icuid,
#                        'outpatient':outpatient_duration,
#                        'emergency':emeregency_duration,
#                        'discharge':discharge_duration,
#                        'inpatient_icu':inpatient_duration,
#                        'icu_stay':icu_duration}
#             stage_durations.loc[len(stage_durations)] = durations
            
#         elif icuids.shape[0] == 1:
#             icuid = icuids[0]
#             icu_stay = admission[admission.icustay_id == icuid]
#             icu_tokens = icu_stay
#             icu_duration = (icu_stay['time'].iloc[-1] - icu_stay['time'].iloc[0]).total_seconds()/(86400)
#             tokens = {'subject_id':pid,
#                        'hadm_id':hid,
#                        'icu_id':icuid,
#                        'outpatient':outpatient_tokens.shape[0],
#                        'emergency':emeregency_tokens.shape[0],
#                        'discharge':discharge_tokens.shape[0],
#                        'inpatient_icu':inpatient_tokens,
#                        'icu_stay':icu_tokens.shape[0]}
#             stage_tokens.loc[len(stage_tokens)] = tokens
#             durations = {'subject_id':pid,
#                        'hadm_id':hid,
#                        'icu_id':icuid,
#                        'outpatient':outpatient_duration,
#                        'emergency':emeregency_duration,
#                        'discharge':discharge_duration,
#                        'inpatient_icu':inpatient_duration,
#                        'icu_stay':icu_duration}
#             stage_durations.loc[len(stage_durations)] = durations
#         else:     
#             for idx, icuid in enumerate(icuids) :
#                 if idx == 0:
#                     icu_stay = admission[admission.icustay_id == icuid]
#                     icu_tokens = icu_stay
#                     icu_duration = (icu_stay['time'].iloc[-1] - icu_stay['time'].iloc[0]).total_seconds()/(86400)
#                     tokens = {'subject_id':pid,
#                                'hadm_id':hid,
#                                'icu_id':icuid,
#                                'outpatient':outpatient_tokens.shape[0],
#                                'emergency':emeregency_tokens.shape[0],
#                                'discharge':discharge_tokens.shape[0],
#                                'inpatient_icu':inpatient_tokens,
#                                'icu_stay':icu_tokens.shape[0]}
#                     stage_tokens.loc[len(stage_tokens)] = tokens
#                     durations = {'subject_id':pid,
#                                'hadm_id':hid,
#                                'icu_id':icuid,
#                                'outpatient':outpatient_duration,
#                                'emergency':emeregency_duration,
#                                'discharge':discharge_duration,
#                                'inpatient_icu':inpatient_duration,
#                                'icu_stay':icu_duration}
                    
#                     stage_durations.loc[len(stage_durations)] = durations
#                 else:
#                     icu_stay = admission[admission.icustay_id == icuid]
#                     icu_tokens = icu_stay
#                     icu_duration = (icu_stay['time'].iloc[-1] - icu_stay['time'].iloc[0]).total_seconds()/(86400)
#                     tokens = {'subject_id':pid,
#                                'hadm_id':hid,
#                                'icu_id':icuid,
#                                'outpatient':np.nan,
#                                'emergency':np.nan,
#                                'discharge':np.nan,
#                                'inpatient_icu':np.nan,
#                                'icu_stay':icu_tokens.shape[0]}
#                     stage_tokens.loc[len(stage_tokens)] = tokens
#                     durations = {'subject_id':pid,
#                                'hadm_id':hid,
#                                'icu_id':icuid,
#                                'outpatient':outpatient_duration,
#                                'emergency':emeregency_duration,
#                                'discharge':discharge_duration,
#                                'inpatient_icu':np.nan,
#                                'icu_stay':icu_duration}
#                     stage_durations.loc[len(stage_durations)] = durations

        
# stage_tokens.replace(np.nan,0.0,inplace=True)
# stage_durations.replace(np.nan,0.0,inplace=True)

# stage_tokens['inpatient_only'] = stage_tokens.apply(lambda x: abs((x.inpatient_icu - x.icu_stay)) if (x.icu_stay != np.nan) and x.icu_stay >0 else x.inpatient_icu - 0, axis=1)
# stage_durations['inpatient_only'] = stage_durations.apply(lambda x: abs((x.inpatient_icu - x.icu_stay)) if (x.icu_stay != np.nan) and x.icu_stay >0 else x.inpatient_icu - 0, axis=1)

# stage_tokens['total'] = stage_tokens.iloc[:,3:-1].sum(axis=1)
# stage_durations['total'] = stage_durations.iloc[:,3:-1].sum(axis=1)

In [18]:
# counts = []
# for pid in tqdm(shard.subject_id.unique()):
#     hids= shard[shard.subject_id == pid].hadm_id.unique()
#     hids = hids[np.isnan(hids) == False].shape[0]
#     counts.append(hids)
# plt.title('Admissions distribution')
# plt.hist(counts,bins=25)

# plt.show()

# counts = []
# for pid in tqdm(shard.subject_id.unique()):
#     icuids= shard[shard.subject_id == pid].icustay_id.unique()
#     icuids= icuids[np.isnan(icuids) == False].shape[0]
#     counts.append(icuids)
# plt.title('ICU admission distribution')
# plt.hist(counts,bins=50)
# plt.show()

In [20]:
# stage_tokens.groupby('subject_id').mean().mean()#groupby('subject_id').mean().mean()

In [21]:
# stage_durations.groupby('hadm_id').mean().mean()

In [22]:
# stage_durations.groupby('subject_id').mean().mean()

In [23]:
# stage_tokens.mean()

In [24]:
# stage_durations.mean()

In [25]:
# stage_durations[stage_durations.icu_stay>0].mean()

In [26]:
# stage_tokens

In [27]:
# stage_durations

In [ ]:
# # over all dataset information collection
# medication = []
# for file in tqdm(os.listdir(mimic_data_path)):
#     part = pd.read_parquet(os.path.join(mimic_data_path,file))
#     med = list(part[part.medication.isna() == False].medication.unique())
#     medication.extend(med) 

    
# medication = list(set(medication))

# medication_names = pd.DataFrame({'original_name': sorted(medication)})
# medication_names['clean'] = medication_names.original_name.apply(lambda x: x.strip().lower())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub('\*nf\*|\*n f\*|\*n\sf|\*nf','',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub('\*+\snon-formulary\s*\*+|\*non-formulary\*','',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub('^\d$','',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub('\(brand name\)','',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(cap|capsule|tab|tabs|tablet|bag|capsules|ophth|opthalmic|packet|pkt|drops|soln|solution|cream|ointment|oint|vial|cup|supp|gelcaps|oral|rinse|drug|brand|concentrated|tablets|pack|combo|liquid|susp|susp.|syr|pak|sol|opth|eye|syrup|name|only|nasal|spray|inhalation|inhal|inh|or|for|human|placeb|placebo|self|administering|medication|regular|kwikpen|sugar|free|table|cvicu|reversal|protocol|supps|adult|emergency|inhaler|suspension|med|tunneled|access|line|bulk|iv|caps|(28)|enablex|(21)|gummies|gummy|sore|throat|lozenge|phenol|relief|dual|benz-men|salmon|test|dose|film|once|daily|chewable|rectal|buffered|inhaled|interventional|pulmonary|use|irrigation|glacial)\b','',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'(\d+)\s*mg', r'\1mg',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'(\d+)\s*ml', r'\1ml',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'(\d+)\s*%', r'\1%',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\s+', r' ',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'(\s*/\s*)','/', x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(alb|albu|album|albumi)\b','albumin',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(albut|albute|albutero)\b','albuterol',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(aspi|aspir|aspiri)\b','aspirin',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(acety|acetyl|acetylcy|acetylc|acetylcys|acetylcyste|acetylcystei|acetylcystein)\b','acetylcysteine',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(art|arti|artif|artifi|artific|artifici)\b','artificial',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(abacav)\b','abacavir',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(acet|aceta|acetam|acetami|acetamin|acetamiinophen|acetamino|acetaminop|acetaminoph)\b','acetaminophen',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(acetazola|acetazolami)\b','acetazolamide',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(add|adde|adder|addera|adderal)\b','adderall',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(altepl|altepla|alteplas)\b','alteplase',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(tea|tear)\b','tears',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(azithr|azithro|azithromy|azithromyc|azithromyci)\b','azithromycin',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(baclo|baclof)\b','baclofen',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(benzon|benzona)\b','benzonatate',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(alumin|aluminu)\b','aluminum',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(aci)\b','acid',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(amin|amino|aminoc|aminoca|aminocap|aminocapr|aminocapro|aminocaproi)\b','aminocaproic',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(amioda)\b','amiodarone',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(amox|amoxic|amoxicill)\b','amoxicillin',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(amph|amphe|amphet|ampheta|amphetam|amphetami|amphetamin)\b','amphetamine',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(ampicill)\b','ampicillin',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(anas|anast|anastr|anastrazole|anastro)\b','anastrozole',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(arip|aripip)\b','aripiprazole',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(balsa|balsal)\b','balsalazide',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(bis|bisac|bisaco|bisacody)\b','bisacodyl',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(brimonidin)\b','brimonidine',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(akwa tears)\b','akwa',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(venl|venlaf|venlafa|venlafax|venlafaxi|venlafaxin)\b','venlafaxine',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(valp|valpr|valproat)\b','valproate',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(valga|valgan|valganc|valganci|valgancic|valgancicl|valganciclo|valganciclov|valganciclovi)\b','valganciclovir',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(tret|tretin)\b','tretinoin',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(tram|trama)\b','tramadol',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(well|wellb|wellbut)\b','wellbutrin',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(xal|xalat)\b','xalatan',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(tim|timo|timolo)\b','timolol',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(tes|tessa)\b','tessalon',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(sulfam|sulfame|sulfamet|sulfameth/trim|sulfameth/trimeth|sulfameth/trimetho|sulfameth/trimethopr|sulfameth/trimethopri)\b','sulfameth/trimethoprim',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(sucr|sucra|sucral|sucralf)\b','sucralfate',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(seve|sevel|sevela|sevelam|sevelame)\b','sevelamer',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(tacr|tacro|tacrol|tacroli|tac)\b','tacrolimus',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(quet|quetiap|quetiapin)\b','quetiapine',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(propr|propra|propran|propranol|propranolo)\b','propranolol',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(predn|predni|prednis|predniso|prednisol)\b','prednisolone',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(pota|potas|potass|potassi|potassiu|potassium)\b','potassium',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(phy|phyt|phyto|phyton|phytona|phytonad|phytonadi)\b','phytonadione',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(pheno|phenob|phenoba|phenobar|phenobarb|phenobarbi|phenobarbit)\b','phenobarbital',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(permet|permeth)\b','permethrin',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(pant|panto|pantopr|pantopra|pantoprazo)\b','pantoprazol',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(pang|pange|pangesty|pangestym)\b','pangestyme',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(pan|panc|pancre)\b','pancrease',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(oxyme|oxymet|oxymeta)\b','oxymetazoline',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(oxyco|oxycod|oxycodo|oxycodon)\b','oxycodone',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(oxcar|oxcarb|oxcarba|oxcarbaz)\b','oxcarbazepin',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(ond|onda|ondan|ondans|ondanse|ondanset|ondansetr|ondansetro)\b','ondansetron',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(ola|olan|olanz|olanza|olanzap|olanzapi)\b','olanzapine',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(ny|nys|nyst|nysta|nystat|nystati)\b','nystatin',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(nitro|nitrof|nitrofur|nitrofuran|nitrofurant)\b','nitrofurantoin',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(su|sul|sulf|sulfa|sulfat)\b','sulfate',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(morp|morph|morphi|morphin)\b','morphine',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(moda|modaf|modafi|modafin|modafini)\b','modafinil',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(meth|metha|methad|methado)\b','methadone',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(mes|mesa|mesal|mesalam|mesalami|mesalamin)\b','mesalamine',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(mu|mucin|mucine)\b','mucinex',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(mycoph|mycophen|mycopheno|mycophenol|mycophenolat)\b','mycophenolate',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(mupi|mupir|mupiro|mupiroci)\b','mupirocin',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(multiv|multivi|multivit|multivitam|multivita|multivitami)\b','mupirocin',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(neph|nephr|nephro|nephrocap)\b','nephrocaps',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(neomy|neo)\b','neomycin',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(nadol)\b','nadolol',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(mov)\b','movantik',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(miner)\b','mineral',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(micon)\b','miconazole',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(metro)\b','metrocream',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(metoprol|metoprolo)\b','metoprolol',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(methylp|methylphenid)\b','methylphenidate',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(blu)\b','blue',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(methylin|methylnal)\b','methylnaltrexone',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(metho|methotr|methotre|methotrexat)\b','methotrexate',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(methadon)\b','methadone',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(metform|metformi)\b','metformin',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(met|metam)\b','metamucil',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(mesna)\b','mesnex',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(mesala)\b','mesalamine',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(merc|merca)\b','mercaptopurine',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(mepe|meper|meperi)\b','meperidine',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(megest)\b','megestrol',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(mecl|mecli)\b','meclizine',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(maa|maal|maalo)\b','maalox',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(lur|luras)\b','lurasidone',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(losar)\b','losartan',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(lora|loraz|loraze)\b','lorazepam',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(loper)\b','loperamide',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(lithi|lithiu)\b','lithium',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(ca|car|carbo|carbon|carbona|carbonat)\b','carbonate',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(liragl)\b','liraglutide',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(j|jel|jell)\b','jelly',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(lid|lido|lidoc|lidoca|lidocai|lidocain)\b','lidocaine',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(levs)\b','levsin',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(levot|levoth|levothy|levothyr|levothyro|levothyrox)\b','levothyroxine',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(levofl)\b','levofloxacin',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(levocarn)\b','levocarnitine',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(levo)\b','levobunolol',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(levet|levetir|levetirace)\b','levetiracetam',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(lev|levalb|leva)\b','levalbuterol',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(leupr)\b','leuprolide',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(les|lesc)\b','lescol',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(lef|lefl|leflu|leflun|lefluno)\b','leflunomide',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(lata|latano)\b','latanoprost',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(lans|lanso|lansop|lansopraz)\b','lansoprazole',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(lamo|lamot|lamotr|lamotrig)\b','lamotrigine',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(lamiv|lamivud)\b','lamivudine',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(lam|lami)\b','lamictal',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(lactul|lactulo)\b','lactulose',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(labe|labet)\b','labetalol',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(ketorola)\b','ketorolac',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(ket|ketam|ketamin)\b','ketamine',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(iso|isos|isoso|isosor|isosorb|isosorbi|isosorbid)\b','isosorbide',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(din|dini|dinitr|dinitra|dinitrat)\b','dinitrate',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(m|mo|mon|mono|monon|mononi|mononit|mononitr|mononitra|mononitrat)\b','mononitrate',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(irb|irbe|irbes|irbesart)\b','irbesartan',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(ipr|ipra|iprat)\b','ipratropium',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(ins|insu|insul|insuli)\b','insulin',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(inde|inder)\b','inderal',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(imod)\b','imodium',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(imd|imdu)\b','imdur',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(imip)\b','imipenem',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(ibu|ibup|ibupro)\b','ibuprofen',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(hyo|hyos|hyosc|hyoscyam)\b','hyoscyamine',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(hydroxyp)\b','hydroxyprogesterone',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(hydromo|hydromorp|hydromorph|hydromorphon)\b','hydromorphone',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(hydrocor|hydrocort|hydrocortis)\b','hydrocortisone',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(hyd|hydr|hydrhydrocort|hydroco)\b','hydrocodone',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(humu|humul)\b','humulin',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(hep|hepa|hepar|hepari)\b','heparin',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(halop|halope|haloperi)\b','haloperidol',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(gu|guai|guaif|guaife|guaifene|guaifenes)\b','guaifenesin',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(goly)\b','golytely',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(glycopyrr|glycopyrrolat)\b','glycopyrrolate',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(gly|glybu|glybur|glyburid)\b','glyburide',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(gluca)\b','glucagon',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(glipi|glipiz|glipizid)\b','glipizide',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(genta)\b','gentak',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(gastr)\b','gastrocrom',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(gala|galant|galanta)\b','galantamine',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(gaba|gabapen)\b','gabapentin',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(furo|furos|furosemi|furose|furosemid)\b','furosemide',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(fluva)\b','fluvastatin',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(flutic)\b','fluticasone',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(fluoromet)\b','fluorometholone',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(flucytos)\b','flucytosine',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(flon|flona|flonas)\b','flonase',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(fio|fior|fiori)\b','fioricet',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(fin|finast|finaste)\b','finasteride',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(fil)\b','filgrastim',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(fexofe|fexofen)\b','fexofenadine',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(ferrou)\b','ferrous',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(fergon)\b','ferocon',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(fenof|fenofi|fenofib)\b','fenofibrate',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(felod|felodi|felodip)\b','felodipine',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(famot)\b','famotidine',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(fam|famc|famci|famcic)\b','famciclovir',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(exe|exem|exemes)\b','exemestane',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(estradi)\b','estradiol',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(estra)\b','estrace',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(eso|esomep)\b','esomeprazole',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(escit|escita|escital)\b','escitalopram',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(ery-|erythr|erythro|erythrom|erythromyc)\b','erythromycin',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(ergo|ergocal|ergocalc|ergocalcif)\b','ergocalciferol',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(epo|epoe|epoet)\b','epoetin',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(epip)\b','epipen',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(ep|epi|epin|epine|epinep|epineph|epinephr|epinephri|epinephrin)\b','epinephrine',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(enox|enoxapa|enoxa|enoxap|enoxapar|enoxapari)\b','enoxaparin',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(emt|emtri|emtrici|emtricita)\b','emtricitab',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(efav)\b','efavirenz',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(econ)\b','econazole',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(doxycy)\b','doxycycline',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(donna)\b','donnatal',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(do|doc|docu|docus|docusa|docusat)\b','docusate',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(s|so|sod|sodi|sodiu)\b','sodium',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(diva|dival|divalp|divalpr|divalpro|divalproe)\b','divalproex',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(dipy)\b','dipyridamole',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(dilti|diltia|diltiaz|diltiaze)\b','diltiazem',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(dil|dila)\b','diltiazem',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(digox)\b','digoxin',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(dextr)\b','dextran',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(dex|dexa|dexametha|dexam|dexame|dexamet|dexameth|dexammetha|dexamethas|dexamethaso|dexamethason)\b','dexamethasone',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(depa|depak)\b','depakene',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(denos)\b','denosumab',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(de)\b','debrox',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(cycloph|cyclopho|cyclophos)\b','cyclophosphamide',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(cyclo|cyclobe|cyclobenza|cyclobenzapri)\b','cyclobenzaprine',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(cyano|cyanoc|cyanoco|cyanocob)\b','cyanocobalamin',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(cre)\b','creatine',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(cov)\b','covid',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(combiv)\b','combivent',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(comb|combi)\b','combigan',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(colchi)\b','colchicine',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(coenz)\b','coenzyme',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(clotrima)\b','clotrimazole',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(clora|cloraz)\b','clorazepate',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(clopidog|clopidogrel)\b','clorazepate',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(clinda|clindagel|clindamyci)\b','clindamycin',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(cisat)\b','cisatracurium',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(ciprofl|ciproflox)\b','ciprofloxacin',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(cil|cilo|cilos|cilost|cilosta|cilostaz)\b','cilostazol',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(chole)\b','cholecalciferol',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(chlorh)\b','chlorhexidine',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(chlor)\b','chloral',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(cet|ceti|cetir|cetiriz|cetirizin)\b','cetirizine',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(cep|cepa|cepac)\b','cepacol',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(cepas)\b','cepastat',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(cefpodo|cefpodoxim)\b','cefpodoxime',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(cefe)\b','cefepime',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(carb|carbam|carbamaze|carbamazep|carbamazepin)\b','carbamazepine',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(capt|capto|captop|captopr)\b','captopril',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(gluc|gluco)\b','gluconate',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(chlori)\b','chloride',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(calc|calci|calcit|calcitoni)\b','calcitonin',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(buta|butalbit|butalbita)\b','butalbital',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(buspi|buspiro)\b','buspirone',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(bupr|bupro|buprop|bupropi|bupropio)\b','bupropion',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(bup|bupreno|buprenor|buprenorp)\b','buprenorphine',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(bu|bum|bume|bumet|bumeta|bumetan|bumetanid)\b','bumetanide',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(bude|bude025|budes|budeso|budeson)\b','budesonide',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(brinz)\b','brinzolamide',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(bisop|bisopro|bisopr)\b','bisoprolol',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(bic|bica|bicalu)\b','bicalutamide',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(betadi)\b','betadine',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(ato|atom|atomox)\b','atomoxetine',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(325)\b','325mg',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\b(asp)\b','aspirin',x).strip())


# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'[*\,]', '',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\(\s*(.*?)\s*\)', r'(\1)',x).strip())
# medication_names['clean'] = medication_names.clean.apply(lambda x: re.sub(r'\(\)', r'',x).strip())

# medication_names['counts'] = medication_names.clean.apply(lambda x:len(x))
# medication_names.original_name = medication_names.original_name.replace('',None)
# medication_names.clean = medication_names.clean.replace('','UNK')
# medication_names.clean = medication_names.clean.replace('c','UNK')
# medication_names.clean = medication_names.clean.replace('d','UNK')
# medication_names.clean = medication_names.clean.replace('t','UNK')
# medication_names.clean = medication_names.clean.replace('al','UNK')
# medication_names.clean = medication_names.clean.replace('h1','UNK')
# medication_names.clean = medication_names.clean.replace('ni','UNK')
# medication_names.clean = medication_names.clean.replace('ru','UNK')
# medication_names.clean = medication_names.clean.replace('zz','UNK')
# medication_names.clean = medication_names.clean.replace('aq','UNK')
# medication_names.clean = medication_names.clean.replace('ce','UNK')
# medication_names.clean = medication_names.clean.replace('ve','UNK')
# medication_names.clean = medication_names.clean.replace('zz','UNK')

In [ ]:
# over all dataset information collection
# microbiology = []
# for file in tqdm(os.listdir(mimic_data_path)):
#     part = pd.read_parquet(os.path.join(mimic_data_path,file))
#     micro = part[part.code.str.startswith('MICROBIOLOGY')]
#     microbiology.append(micro) 


# microbiology = pd.concat(microbiology, ignore_index=True)

In [384]:
# values = ['HOLD.  DISCARD GREATER THAN 4 HOURS OLD.',
#  'HOLD.  DISCARD GREATER THAN 24 HRS OLD.', 
#  'HOLD.',
#  'HOLD.  SPECIMEN TO BE HELD 48 HOURS AND DISCARDED.',
#  'HOLD.  DISCARD AFTER THREE DAYS .',
#  'UNABLE TO REPORT', 
#  'UNABLE', 'UNABLE TO REPORT,QNS',
#  'UNABLE TO RESULT', 
#  'UNLABE TO REPORT',
#  'UNABLE TO RPEORT', 
#  'UNABLE TO REPORT INR QNS',
#  'UNABLE TO REPORT PTT QNS',
#  'UNABLE TO REPORT, SPECIMEN GROSSLY BLOODY', 
#  'UNABLE TO PERFORM',
#  'UNABLE TO REPORT, LAB ERROR', 
#  'UNABLE TO REPORT, SPECIMEN QNS',
#  'UNABLE TO REPORT, HCT GREATER THAN 55', 
#  'UNABLE TO REPORT, QNS',
#  'UNABLE TO REPORT, GROSSLY BLOODY SPECIMEN',
#  ':UNABLE TO REPORT',
#  'ERROR', 
#  'ERROR,DISREGARD PREVIOUS RESULT OF 11.8', 
#  'ERRROR',
#  'EERROR',
#  'DDONE',
#  'DONE', 
#  'DONE.', 
#  'DONE.  DISCARD GREATER THAN 4 HOURS OLD.',
#  "DONE'", 
#  'DONE.  DISCARD GREATER THAN 24 HRS OLD.', 
#  'DONE BY ___',
#  'DONEDONE', 
#  'DONED',
#  'DOEN',
#  'DOE',
#  'SAMPLE CLOTTED',
#  'CANCEL'
#  'CNACEL',     
#  'CANCEL', 
#  'CANCELLED', 
#  'CANCLE', 
#  'CANCELED', 
#  'CANCLLED',
#  'CANCELED BY ___', 
#  'CANCELLE',
#  'CALCEL', 
#  'CALVEL',
#  'PENDING',
#  'CNACEL',
#  'SPECIMEN OLD NOTIFIED ___ @710 ON ___',
#  'SPECIMEN CLOTTED. PREVIOUSLY REPORTED AS 0.9.',
#  'SPECIMEN COMPROMISED. DISREGARD PREVIOUS RESULT OF 3.2.',
#  'SONE',
#  'VOIDED', 
#  'VOID',
#  'UNK',
#  'LUPUS CANCELED',
#  'TEST CANCELLED BY ___',
#  '-',
#  'CLOTTED',
#  'INDETERMNATE',
#  'INDERTERMINATE',
#  'INDETERMNATE',
#  'LAB ERROR',
#  '‰',
#  'DOINE',
#  'DINE',
#  'CORRECTED RESULT'
#  'DK',
#  'SL',
#  'DARK',
#  'NOT DONE',
#  ':ERROR',
#  'NotDone',
#  'INVALID',
#  ''
# ]

In [290]:
# not_labs = [50807, 50812, 50829, 50845, 50886, 50887, 50888, 50897, 50919,
#             50923, 50932, 50933, 50934, 50947, 50955, 50979, 50984, 50985,
#             51038, 51056, 51103, 51107, 51129, 51571, 51591, 51599, 51600,
#             51601, 51602, 51603, 51604, 51608, 51612, 51671, 51678, 51698,
#             51699, 51700, 51702, 51703, 51706, 51712, 51717, 51718, 51719,
#             51720, 51727, 51752, 51757, 51759, 51760, 51771, 51796, 51806,
#             51827, 51828, 51830, 51831, 51839, 51901, 51905, 51906, 51907,
#             51924, 51953, 51955, 51978, 51993, 51995, 51997, 51998, 52014,
#             52016, 52023, 52025, 52033, 52036, 52043, 52066, 52067, 52068,
#             52118, 52161, 52186, 52195, 52229, 52230, 52231, 52232, 52233,
#             52234, 52235, 52236, 52237, 52238, 52239, 52240, 52241, 52242,
#             52243, 52244, 52245, 52246, 52247, 52248, 52249, 52250, 52251,
#             52252, 52253, 52254, 52287, 52288, 52289, 52290, 52313, 52314,
#             52315, 52334, 52370, 52371, 52372, 52374, 52392, 52393, 52405,
#             52406, 52412, 52415, 52418, 52419, 52420, 52421, 52422, 52423,
#             53127, 51564, 51597, 51605, 51657, 51658, 51659, 51660, 51661, 
#             51663, 51664, 51665, 51686, 51732, 51733, 51734, 51735, 51736,
#             51737, 51762, 51763, 51764, 51765, 51766, 51767, 51768, 51772,
#             51789, 51817, 51849, 51850, 51851, 51852, 51856, 51857, 51902,
#             51903, 51904, 51908, 51909, 51916, 51939, 51954, 51956, 51970,
#             51971, 51973, 52004, 52005, 52006, 52007, 52008, 52009, 52010,
#             52011, 52012, 52018, 52019, 52020, 52021, 52080, 52081, 52083,
#             52084, 52110, 52136, 52137, 52147, 52148, 52153, 52169, 52191,
#             52194, 52215, 52217, 52317, 52318, 52333, 52394, 52395, 52396,
#             52397, 52398, 52399, 52400, 52401, 52402, 52424, 52425, 52426,
#             52427, 53122, 51662, 50827, 50828, 51509,51513]

In [327]:
# over all dataset information collection
# labs = []
# for file in tqdm(os.listdir(mimic_data_path)):
#     part = pd.read_parquet(os.path.join(mimic_data_path,file))
#     lab = part[part.code.str.startswith('LAB')]
#     lab = lab[lab.text_value.notna()]
#     labs.append(lab) 


# labs = pd.concat(labs, ignore_index=True)

In [12]:
# labs['table'] = labs.code.apply(lambda x: x.split('//')[-1])
# labs['code_type'] = labs.code.apply(lambda x: x.split('//')[-1])
# labs.code = labs.code.apply(lambda x: '//'.join(x.split('//')[:-1]))

In [189]:
# labs_metadata = labs_metadata.rename(columns={'itemid (omop_source_code)':'itemid'})

# lab_label = dict(zip(labs_dimension['itemid'], labs_dimension['label']))
# lab_fluid = dict(zip(labs_dimension['itemid'], labs_dimension['fluid']))
# lab_category = dict(zip(labs_dimension['itemid'], labs_dimension['category']))
# lab_description = dict(zip(labs_metadata['itemid'], labs_metadata['omop_concept_name']))
# lab_frequency = dict(zip(labs_metadata['itemid'], labs_metadata['labevents_row_count']))

In [14]:
# labs['lab_label'] =  labs.lab_itemid.map(lab_label)
# labs['lab_fluid'] =  labs.lab_itemid.map(lab_fluid)
# labs['lab_category'] =  labs.lab_itemid.map(lab_category)
# labs['lab_description'] =  labs.lab_itemid.map(lab_description)
# labs['lab_frequency'] =  labs.lab_itemid.map(lab_frequency)

# labs.lab_label = labs.lab_label.apply(lambda x: x.split(', ')[0] if type(x) == str 
#                                         and ((x.split(', ')[-1] in list(labs_metadata.fluid.unique())) 
#                                         or (x.split(', ')[-1] in ['Body Fluid', 'Other Fluid'])) 
#                                         else x)

# labs.code = labs.apply(lambda x: '//'.join([x.code,x.lab_label]) if x.code.startswith('LAB//') else x.code,axis= 1)




In [47]:
# labs_metadata[labs_metadata.itemid.isin(a) == False]

In [48]:
# int(labs_metadata.labevents_row_count.sum())

In [326]:
# labs_metadata = labs_metadata[labs_metadata.labevents_row_count.isna() == False]
# labs_metadata = labs_metadata.sort_values('labevents_row_count',ascending=False)
# a = labs_metadata[labs_metadata.sssom_comment == 'Not a lab test'].itemid.unique()
# labs_metadata = labs_metadata[labs_metadata.itemid.isin(a) == False]
# print(int(labs_metadata.labevents_row_count.sum()))
# labs_metadata['percent'] = labs_metadata.labevents_row_count.apply(lambda x: (x/116770303)*100)
# labs_metadata.iloc[:200].percent.sum()

In [385]:
# labs = labs[labs.text_value.isin(values) == False]
# labs = labs[labs.lab_itemid.isin(not_labs) == False]

In [479]:
# lab_values = pd.DataFrame(data=labs.text_value.unique(), columns=['original_values'])
# lab_values['clean'] = lab_values['original_values'].apply(lambda x: x.lower())
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(hegative|neative|negarive|megative|neg|negaitive|negaitve|negatative|negativ|negativae|neg.|negativie|negatove|negtive|negotive|negtaive|nehative|negstive|negayive|negavite|negavite|negatvie|negative.|nrgative)\b','negative',x).strip())
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(negativefoe|negativefor|negativefot)\b','negative for',x).strip())
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(negativethick)\b','negative thick',x).strip())
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(negativethin)\b','negative thin',x).strip())
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(noraml|norma)\b','normal',x).strip())
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(notdone)\b','not done',x).strip())
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(occ|occ.|occas|occassional|occasional.)\b','occasional',x).strip())
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(pos|potive|postive|postitive)\b','positive',x).strip())
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(notdet|notdetected)\b','not detected',x).strip())
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(positve|positi\?ve|positi?ve|positive\.)\b','positive',x).strip())
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(or|oran|ornage)\b','orange',x).strip())
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(refra)\b','refractometer',x).strip())
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(yel|yell|uellow|yelllow|yello|yrllow|tellow)\b','yellow',x).strip())
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub('\.(?=\s|$)','',x).strip())
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\.\.+', '.', x).strip())
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'^:', '', x).strip())
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'^\'', '', x).strip())
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\s*[cC]$', '', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(nad|snd)\b','and',x).strip())
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(or)\b','and',x).strip())
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(prresence)\b','presence',x).strip())
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(smear)\b','smears',x).strip())
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(exracellular|extracellular)\b','exra cellular',x).strip())
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(reviewed|reviewe|review|reviwed)\b','',x).strip())
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(tick)\b','thick',x).strip())
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(extracellularforms)\b','extracellular forms',x).strip())
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(extra)(cellular)\b', r'\1 \2', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(extra)(celluar)\b', r'\1 \2', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(celluar)\b','cellular',x).strip())
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(intra)(cellular)\b', r'\1', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(exra)\b','extra',x).strip())
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(intra)\s+and\s+(extra)\b', r'\1/\2', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(prasite|parasits|parasite)\b','parasites',x).strip())
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(prasite|parasits|parasite)\b','parasites',x).strip())
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(re|redd|rep|reare)\b','red',x).strip())
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(ha|haz)\b','hazy',x).strip())
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(dk)(red)\b', r'red', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(dk)(a)\b', r'amber', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(dk)(aml)\b', r'amber', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(dk)(mb)\b', r'amber', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(dk)(rmb)\b', r'amber', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(dl)(amb)\b', r'amber', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(dk)(am)\b', r'amber', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(dk)(amb)\b', r'amber', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(dk)(amber)\b', r'amber', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(dk)(brwn)\b', r'brown', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(dk)(brown)\b', r'brown', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(dk)(green)\b', r'green', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(dk)(green)\b', r'green', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(dk)(yel)\b', r'yellow', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(dk)(yello)\b', r'yellow', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(dk)(yellow)\b', r'yellow', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(dr)(amb)\b', r'amber', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(d)(yellow)\b', r'yellow', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(drk)(amb)\b', r'amber', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(drk)(amber)\b', r'amber', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(dark)(amber)\b', r'amber', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(dark)(yellow)\b', r'yellow', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(d)(amber)\b', r'amber', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(br|bro|brow)\b','brown',x).strip())
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(c|cl|clo|clou|cldy)\b','cloudy',x).strip())
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(drk)(mb)\b', r'dark amber', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(d)(yellow)\b', r'yellow', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(drk)(yellow)\b', r'yellow', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(kd)(amb)\b', r'amber', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(lt)(amb)\b', r'amber', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(lt)(amber)\b', r'amber', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(lt)(brown)\b', r'brown', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(sl)(hazy)\b', r'hazy', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(sl)(cloudy)\b', r'cloudy', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(sl)(cldy)\b', r'cloudy', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(sma)\b','small',x).strip())
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(tra)\b','trace',x).strip())
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(defficient|deficinet)\b','deficient',x).strip())
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'^\.(\w+)', r'\1', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'verified', '', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'--+', '-', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'=', '-', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'=', '-', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'greater than\s*(\d+)', r'>\1', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'less than\s*(\d+)', r'>\1', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'^\?(\w+)', r'\1', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\\+', '', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'€', '', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'refractometer', '', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'cc', '', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r' notified ___ 7:55am ___$', '', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r' called to ___ at 12:26 on ___$', '', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r' 150 is highest meeasured ptt: notified ___ at 4:30pm on ___$', '', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'150:150 is', '150', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'>-([0-9.]+)', r'>=\1', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'> 1.035', r'>1.035', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'1.005l', r'1.005', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'1.000:by', r'1.000', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r' notified ___ @1:00pm$', '', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r':corrected results:previously reported as 555:notify ___$', '', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'1.000:by', r'1.000', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'…12.1', r'12.1', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r' , if high clinical suspicion of malaria repeat screen every 12-24 hours for 3 consecutive days$', '', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'few]', r'few', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'few\'', r'few', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'3.-5', r'3-5', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'3-5-', r'3-5', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'^3-$', r'3-5', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'0.2 %', r'0.2%', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'4.9', r'4.9%', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'  by dilution$', '', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'6-10-', '6-10', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'6-106-10', '6-10', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'<-([0-9.]+)', r'>=\1', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'23.6,checked forclot$', '23.6', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r' specimen lipemic$', '', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'1.0188.0', '1.0188', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'1.0056.5', '1.0056', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'1.0056.5', '1.0056', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'1.0218.0', '1.0218', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'1.0305.0', '1.0305', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'0.-2', '0-2', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(am|amb|amner|anber)\b','red',x).strip())
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r' reviewd""if high clinincal suspicion of malaria$', '', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'negative, thin and thick smears', 'negative thin and thick smears', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'negative, thin and thick smears', 'negative thin and thick smears', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'negative thin and thick smears', 'negative for thin and thick smears', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'negative thin and thick smears', 'negative for thin and thick smears', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'o-2', '0-2', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'oasional', 'occasional', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'positive, 0.7% parasitemia', '0.7% parasitemia', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'rare/', 'rare', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'&', 'and', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'1.0.14', '1.014', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'1.0.15', '1.015', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'1.0.16', '1.016', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'1.0.17', '1.017', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'1.0.18', '1.018', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'1.0.27', '1.027', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(ava|avai|avail)\b','available',x).strip())
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'negative thick smears', 'negative for thick smears', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'negative on thick smears', 'negative for thick smears', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'negative on thin smears', 'negative for thin smears', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'negative thick and thin smears', 'negative for thick and thin smears', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'negative for intra/extra cellular forms,thin and thick smears', 'negative for intra/extra cellular forms, thin and thick smears', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'negative for intra/extra cellular forms, thick and thin smears', 'negative for intra/extra cellular forms, thin and thick smears', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'negative for thick and thin smears', 'negative for thin and thick smears', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(la|larg|lg|lge|lrg)\b','large',x).strip())
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'0-2coarse', '0-2 coarse', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'0-2,coarse', '0-2 coarse', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\"abnormal urine color-interpret dipstick with caution\"', 'abnormal urine color-interpret dipstick with caution', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\"abnormal urine color-interpret dipstick with caution.\"', 'abnormal urine color-interpret dipstick with caution', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(sm)\b','small',x).strip())
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(lg|lge)\b','large',x).strip())
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(tr)\b','trace',x).strip())
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(nromal)\b','normal',x).strip())
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'negative for intra orange extra cellular forms', 'negative', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'negative for intra orange extra cellular forms on thin smears', 'negative', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'negative for intra orange extra cellular', 'negative', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'negative for intra/extra cellular forms', 'negative', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'negative for intra/extra cellular parasites', 'negative', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'negative for intra/extra cellular organisms', 'negative', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'negative for the presence of microfilariae', 'negative', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'negative for thick smears', 'negative', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'negative for thin smears', 'negative', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'negative for thin and thick', 'negative', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'negative on thin smears', 'negative', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'negative smears', 'negative', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'negative, thin and thick smears', 'negative', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'occasional small clumps', 'occasional', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'occasional sodium urate crystals', 'occasional', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'thick smears  negative', 'negative', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'thick smears negative', 'negative', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'sodium urates rare', 'rare', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'rare coarse granular cast', 'rare', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'positive 1.9', 'positive', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'positive for intacellular', 'positive', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'positive for intra/extra cellular forms', 'positive', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'positive for anaplasma', 'positive', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'0-2 3-5', '0-2', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'0-2 coarse', '0-2', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'0-2 and fine granular casts', '0-2', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'0-2 fine granular casts', '0-2', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'0-2 granular casts', '0-2', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'0-2\'one\(1\) large clump', '0-2', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'0-2 trans', '0-2', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'0-2\+', '0-2', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'0-2f', '0-2', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'0-2r', '0-2', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'0.1% parasitemia', 'positive', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'0.2% parasitemia', 'positive', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'0.5% parasitemia', 'positive', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'0.6% parasitemia', 'positive', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'0.7% parasitemia', 'positive', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'0.8% parasitemia', 'positive', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'0.9% parasitemia', 'positive', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'1.3% parisitemia', 'positive', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'4% parasitimia', 'positive', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'4.9% parasitemia', 'positive', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'5.1% parasetemia', 'positive', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'7.4% parasitemia', 'positive', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'35:<35', '35', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'3-5 coarse granular casts', '3-5', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'3-5 fine granular casts', '3-5', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'3-5 occasional clump', '3-5', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'3-53o', '3-5', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'6-10 coarse granular casts', '6-10', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'boderline positive by drvvt', 'positive', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'boderline', 'positive', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'borderline positive for pt mixing study', 'positive', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'present', 'positive', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'not detected', 'negative', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'i/e', 'internal and external', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'\b(m0d|mo|md|mod|mod\-)\b','moderate',x).strip())
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'moderate-','moderate',x).strip())
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'few small','few',x).strip())
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'few sodium urate crystals','few',x).strip())
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'f-moderate','few',x).strip())
# lab_values['clean'] = lab_values.clean.apply(lambda x: re.sub(r'trichomonas positive', 'positive', x))
# lab_values['clean'] = lab_values.clean.apply(lambda x: x.upper())




# lab_values.clean.unique().shape

(276,)

In [476]:
labs['text_val'] = labs.text_value.map(dict(zip(lab_values.original_values,lab_values.clean))) 

In [477]:
a = labs.groupby('text_val').count().sort_values('time',ascending=False)

In [482]:
b = a.iloc[:50].reset_index().text_val

In [489]:
lab_values = lab_values[lab_values.clean.isin(b)]

In [496]:
lab_values.clean.unique()

array(['___', 'NONE', 'NEGATIVE', 'OCCASIONAL', 'LARGE', '0-2', '6-10',
       'CLEAR', 'YELLOW', 'FEW', 'MODERATE', 'TRACE', 'SMALL', 'NORMAL',
       'ORANGE', '3-5', 'RARE', 'STRAW', '11-20', 'POSITIVE', 'SENT',
       '<1', 'MANY', 'CLOUDY', 'RED', 'D', 'HAZY', '>50', '36.7', '37.1',
       'AMBER', '>1.035', 'BROWN', '>300', '21-50', '10-40', '>=300',
       'PINK', '>=1.035', '0-10', 'INTRA', 'NEEDLE', '>1.030', '>1000',
       'INTERNAL AND EXTERNAL', 'RHOMBOID', '37.2', '40-80', '<1.005',
       '37'], dtype=object)

In [499]:
# lab_values.clean.replace('0-2','1',inplace=True)
# lab_values.clean.replace('6-10','8',inplace=True)
# lab_values.clean.replace('3-5','4',inplace=True)
# lab_values.clean.replace('11-20','15',inplace=True)
# lab_values.clean.replace('<1','0.8',inplace=True)
# lab_values.clean.replace('>50','55',inplace=True)
# lab_values.clean.replace('>1.035','1.05',inplace=True)
# lab_values.clean.replace('>=1.035','1.04',inplace=True)
# lab_values.clean.replace('>300','330',inplace=True)
# lab_values.clean.replace('21-50','35',inplace=True)
# lab_values.clean.replace('10-40','20',inplace=True)
# lab_values.clean.replace('>=300','310',inplace=True)
# lab_values.clean.replace('0-10','5',inplace=True)
# lab_values.clean.replace('>1.03','1.04',inplace=True)
# lab_values.clean.replace('>1.030','1.04',inplace=True)
# lab_values.clean.replace('>1000','1.100',inplace=True)
# lab_values.clean.replace('40-80','60',inplace=True)
# lab_values.clean.replace('<1.005','1.01',inplace=True)









In [502]:
# lab_values['numric'] = pd.to_numeric(lab_values.clean,errors='coerce')

In [507]:
# lab_values = lab_values.reset_index(drop=True)

In [323]:
# lab_values.to_csv('../resources/labs/lab_textual_mapping.csv',index=False)

In [ ]:
ICU categories
['RNTriggerNote',
 'Hemodynamics',
 'Pain/Sedation',
 'Durable VAD',
 'Care Plans',
 'GI/GU',
 'Alarms',
 'Pastoral Care Note',
 'PatientSafetyInitialNote',
 'Skin - Assessment',
 'Swallow Evaluation',
 'General',
 'OT Notes',
 'Centrimag',
 'Case Management',
 'Impella',
 'Toxicology',
 'IABP',
 'MD Progress Note',
 'Cardiovascular (Pacer Data)',
 'Dialysis',
 'Neurological',
 'Respiratory',
 'Adm History/FHPA',
 'Scores - APACHE IV (2)',
 'OB-GYN',
 'PiCCO',
 'Cardiovascular',
 'Tandem Heart',
 'Access Lines - Peripheral',
 'Heartware',
 'Skin - Incisions',
 'ECMO',
 'Pulmonary',
 'Cardiovascular (Pulses)',
 'Labs',
 'Treatments',
 'Scores - APACHE II',
 'Skin - Impairment',
 'NICOM',
 'RDOS',
 'Access Lines - Invasive',
 'Restraint/Support Systems',
 'Routine Vital Signs',
 'Block Charting Note',
 'PA Line Insertion']

# Fluid output categories
# ['Drains', 'Output']

# Infusion categories
['Antibiotics',
 'Nutrition - Supplements',
 'Nutrition - Enteral',
 'Blood Products/Colloids',
 'Medications',
 'Fluids/Intake',
 'Nutrition - Parenteral']


# Procedures categories
['5-Imaging',
 '2-Ventilation',
 '4-Procedures',
 'GI/GU',
 'Access Lines - Peripheral',
 'Access Lines - Invasive',
 '7-Communication',
 'Medications',
 '1-Intubation/Extubation',
 '3-Significant Events',
 '6-Cultures',
 'Dialysis']


In [325]:
cat_proc

['5-Imaging',
 '2-Ventilation',
 '4-Procedures',
 'GI/GU',
 'Access Lines - Peripheral',
 'Access Lines - Invasive',
 '7-Communication',
 'Medications',
 '1-Intubation/Extubation',
 '3-Significant Events',
 '6-Cultures',
 'Dialysis']

In [52]:
print(*['Antibiotics',
 'Nutrition - Supplements',
 'Nutrition - Enteral',
 'Blood Products/Colloids',
 'Medications',
 'Fluids/Intake',
 'Nutrition - Parenteral'])

Antibiotics Nutrition - Supplements Nutrition - Enteral Blood Products/Colloids Medications Fluids/Intake Nutrition - Parenteral
